# P1–P5: shape + spectrum cross-fit ensemble

**One fixed workflow. Attach `fyp-dataset`, enable GPU T4 ×2, then Save Version → Save & Run All.** No previous results ZIP, model upload, Google Drive access, pip installation or mode selection is required. The verified data ZIP can also be attached as input and is extracted after integrity checks.

The current balanced baseline reached 50.2% isolated action macro F1 and 33.2% sign macro F1. Story identity, seated recognition and false commands remain weak. This experiment targets those issues; it does not promise an improved score.

The main workload is **20 × 9-minute optimization jobs on two GPUs**, approximately 90 minutes plus setup, validation inference and export. Two final all-five models follow, within the remaining time. Target wall time: roughly **100–115 minutes** on two T4 GPUs. There are no artificial waits or early-stopping shortcuts. Kaggle queuing/publishing is outside this runtime. A worker/global safety limit can interrupt abnormal runs; incomplete results are explicitly marked.

### What changes

1. A single four-second CNN learns temporal movement shape and four signal-frequency bands, retaining energy features for weak movements. Group normalization and attention pooling reduce dependence on a batch’s people and motion timing.
2. Action and sign identities have separate heads and losses. Class/person/source/posture/event balancing prevents long standing recordings from dominating shorter seated and story examples. Cue/key-press labels remain unverified by video.
3. A small supervised contrastive loss uses matching labels from different **training people** as positives. This is an experimental application of the [supervised contrastive learning objective](https://arxiv.org/abs/2004.11362); the paper does not establish performance on this dataset.
4. Moving-average model weights and validation checkpoints are used. For each outer test person, four models each train on three other people and validate on the remaining person. Average their probabilities. All four non-test people contribute across the ensemble; no member trains on the outer test person.
5. Sign detection and confidence rejection are selected from those four out-of-fold validation people, emphasizing correct-command precision. Test scores never choose weights or thresholds.

The preprocessing, quality masks, covered event IDs, class order and story overlap matching are retained for comparison. No test-person calibration is used. All-five deployment weights have no new independent score. This revision is informed by earlier test results; a fresh session/person is still needed for a final independent claim.


In [ ]:
import time
RUN_STARTED=time.time()
DATA_ROOT=""  # auto-discover the attached fyp-dataset/phase1 folder
ZIP_PATH=""   # optional alternative: attached FYP_Phase1_P01_P05_verified.zip
EXPECTED_ZIP_SHA256="74259c5718d512ffa9cd3bdf63f150badcd6887e0cd2b35f780602e6e13a165c"
EVALUATION_DEADLINE=RUN_STARTED+103*60
TRAINING_DEADLINE=RUN_STARTED+112*60
FINISH_DEADLINE=RUN_STARTED+118*60


## GPU and workspace checks

The intended schedule requires two GPUs. This cell stops before training if two GPUs are unavailable. The notebook keeps the installed Kaggle PyTorch environment and uses [automatic mixed precision](https://docs.pytorch.org/docs/stable/notes/amp_examples.html). Training is blocked on a laptop.


In [ ]:
from pathlib import Path
import os,sys,json,shutil,importlib,importlib.util
assert Path('/kaggle/working').is_dir(), 'Run this notebook on Kaggle. Laptop training is disabled.'
for module in ['torch','numpy','pandas','scipy','sklearn','joblib','matplotlib']:
    assert importlib.util.find_spec(module), f'Missing required Kaggle package: {module}'
import torch
assert torch.cuda.is_available() and torch.cuda.device_count()>=2, 'Enable GPU T4 x2 in Kaggle Settings, then start a fresh run.'
for i in range(2):
    free,total=torch.cuda.mem_get_info(i)
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}; free {free/2**30:.1f} GiB',flush=True)
    assert free>7*2**30, 'GPU memory is occupied. Stop other work and start a fresh session.'
WORK=Path('/kaggle/working')/('FYP_IMPROVED_2H_'+time.strftime('%Y%m%d_%H%M%S',time.gmtime(RUN_STARTED)))
OUT=WORK/'results';OUT.mkdir(parents=True,exist_ok=True)
SOURCE_DIR=WORK/'source';SOURCE_DIR.mkdir(exist_ok=True)
from IPython.display import display,FileLink,Image
import pandas as pd


In [ ]:
SOURCES = {'fyp_common.py': '"""P01-P05 CSI experiments. No training is performed on import.\nResidual baseline inspired by ../Model_Detect-Activity-and-Gestures/cnn.py.\nNew split, quality, normalization, selection, multiscale and checkpoint pipeline.\n"""\nfrom pathlib import Path\nimport os, json, hashlib, random, shutil, zipfile, time, gc, platform\nimport numpy as np\nimport pandas as pd\nfrom scipy.signal import butter, sosfilt, sosfilt_zi, welch\nfrom sklearn.ensemble import ExtraTreesClassifier\nfrom sklearn.metrics import f1_score, classification_report, confusion_matrix, roc_auc_score\nimport joblib\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nVERSION = \'p1p5-balanced-2h-v2\'\nPIDS = [\'P01\',\'P02\',\'P03\',\'P04\',\'P05\']\nACTIONS = [\'walk\',\'sitstand\',\'pickup\',\'lightswitch\',\'drink\',\'answerphone\',\'eat\',\'wipetable\',\'tvremote\',\'fall\',\'chop_food\',\'stir_pot\']\nSIGNS = [\'HELLO\',\'THANKYOU\',\'LAMP\',\'FAN\',\'TELEVISION\',\'COOK\',\'WATER\',\'HOT\',\'HUNGRY\',\'TOILET\',\'CALL\',\'HELP\',\'PAIN\',\'ACCIDENT\']\nFINE = ACTIONS + SIGNS\nSTATES = [\'idle\',\'activity\',\'sign\']\nFS = 50\nMAX_WIN = 6*FS\nHOP = 10  # 0.2 seconds\nCANDIDATES = [dict(name=\'residual4\',kind=\'residual\',seconds=4),\n              dict(name=\'residual6\',kind=\'residual\',seconds=6),\n              dict(name=\'normalized4\',kind=\'residual\',seconds=4,representation=\'window_normalized\')]\nSESSION_DEADLINE = None\n\nclass SessionBudgetReached(RuntimeError):\n    pass\n\nDEFAULTS = dict(epochs=100, patience=8, min_epochs=24, passes_per_epoch=1.,\n                batch_size=256, lr=0.001, weight_decay=0.01, seed=17, seeds=[17,29],\n                val_every=4, tree_estimators=300)\nVALIDATION_PID = {\'P01\':\'P04\',\'P02\':\'P05\',\'P03\':\'P01\',\'P04\':\'P02\',\'P05\':\'P03\'}\n\nPROGRESS_PATH = None\nCURRENT_JOB = \'\'\n\ndef progress(phase):\n    if PROGRESS_PATH is not None:\n        atomic_json(dict(time=time.time(), job=CURRENT_JOB, phase=phase), PROGRESS_PATH)\n\n\n\n\ndef sha256(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for b in iter(lambda:f.read(8*1024*1024),b\'\'): h.update(b)\n    return h.hexdigest()\n\n\ndef atomic_json(obj,path):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_suffix(path.suffix+\'.tmp\')\n    tmp.write_text(json.dumps(obj,indent=2,default=lambda x:x.item() if hasattr(x,\'item\') else str(x)))\n    os.replace(tmp,path)\n\n\ndef verify_data(root):\n    root=Path(root)\n    manifest=json.loads((root/\'manifest.json\').read_text())\n    assert manifest[\'participants\']==PIDS, \'This notebook requires all five real participants.\'\n    for name,info in manifest[\'files\'].items():\n        f=(root/name).resolve()\n        assert f.is_relative_to(root.resolve()),name\n        assert f.is_file() and f.stat().st_size==info[\'bytes\'],f\'Missing/incomplete: {name}\'\n        assert sha256(f)==info[\'sha256\'],f\'Hash mismatch: {name}\'\n    return sha256(root/\'manifest.json\')\n\n\ndef extract_verified(archive, destination, expected_sha=\'\'):\n    archive=Path(archive); destination=Path(destination)\n    if expected_sha: assert sha256(archive)==expected_sha, \'ZIP SHA256 mismatch: download again.\'\n    with zipfile.ZipFile(archive) as z:\n        assert z.testzip() is None,\'Corrupt ZIP: download again.\'\n        base=destination.resolve()\n        for member in z.infolist():\n            p=(base/member.filename).resolve()\n            assert p.is_relative_to(base),\'Unsafe ZIP path\'\n            assert (member.external_attr>>16)&0o170000 != 0o120000,\'Symlink in ZIP\'\n        z.extractall(destination)\n    root=destination/\'phase1\'\n    verify_data(root)\n    return root\n\n\ndef resolve_csi(root,name):\n    p=Path(root)/Path(name).name\n    if not p.exists(): p=Path(str(p)+\'.gz\')\n    assert p.exists(),p\n    return p\n\n\ndef raw_to_stream(path):\n    # No whole-recording fitted mean, PCA or noise-floor normalization.\n    df=pd.read_csv(path)\n    expected=[\'Packet\',\'Timestamp_us\',\'Channel\']+[q for c in range(52) for q in (f\'SC{c}_Re\',f\'SC{c}_Im\')]\n    assert list(df.columns)==expected,f\'Unexpected CSI schema: {path}\'\n    t=df.Timestamp_us.to_numpy(np.int64)\n    assert len(t)>100 and (np.diff(t)>0).all(),f\'Invalid timestamps: {path}\'\n    assert df.Channel.eq(1).all(),f\'Wrong channel: {path}\'\n    v=df.iloc[:,3:].to_numpy(np.float32)\n    assert np.isfinite(v).all(),f\'Nonfinite CSI: {path}\'\n    a=np.hypot(v[:,::2],v[:,1::2]); x=np.log(np.maximum(a,1e-6))\n    rel=(t-t[0])/1e6\n    grid=np.arange(0,rel[-1],0.01)\n    # Resample at 100 Hz before low-pass and decimation to 50 Hz.\n    xg=np.stack([np.interp(grid,rel,x[:,c]) for c in range(52)],axis=1)\n    idx=np.clip(np.searchsorted(rel,grid),1,len(rel)-1)\n    gap=rel[idx]-rel[idx-1]\n    valid=gap<=0.100\n    severe=gap>0.250\n    sos=butter(2,[0.3,18.0],btype=\'bandpass\',fs=100,output=\'sos\')\n    # Reset filter state following a severe gap. Mark warmup as unavailable.\n    cuts=np.r_[0,np.flatnonzero(severe[:-1] & ~severe[1:])+1,len(grid)]\n    hp=np.zeros_like(xg)\n    for a0,b0 in zip(cuts[:-1],cuts[1:]):\n        if b0<=a0: continue\n        zi=sosfilt_zi(sos)[:,:,None]*xg[a0][None,None,:]\n        hp[a0:b0],_=sosfilt(sos,xg[a0:b0],axis=0,zi=zi)\n        valid[a0:min(a0+300,b0)]=False\n    return dict(t=t[0]+np.round(grid[::2]*1e6).astype(np.int64),\n                x=hp[::2].astype(np.float32), valid=valid[::2],\n                severe=severe[::2], raw_first=int(t[0]),raw_last=int(t[-1]),\n                rows=len(t), raw_rate=len(t)/rel[-1], max_gap_ms=float(np.diff(rel).max()*1000))\n\n\ndef infer_posture(r,part):\n    if isinstance(r.get(\'posture\'),str) and r[\'posture\']: return r[\'posture\'],\'recorded\'\n    # Do not invent original story postures. Explicitly report unknown.\n    if part==\'signs_standing\': return \'standing\',\'part_name\'\n    return \'unknown\',\'unrecorded\'\n\n\ndef prepare(root,work):\n    root=Path(root); work=Path(work); work.mkdir(parents=True,exist_ok=True)\n    fingerprint=verify_data(root)\n    log=pd.read_csv(root/\'session_log.csv\')\n    log=log[log.status.eq(\'kept\') & log.pid.isin(PIDS)].sort_values([\'date\',\'time\']).drop_duplicates([\'pid\',\'part\'],keep=\'last\')\n    assert set(log.pid)==set(PIDS)\n    cache=work/\'cache\'; cache.mkdir(exist_ok=True)\n    streams=[]; audit=[]; events=[]\n    for take in log.itertuples():\n        src=resolve_csi(root,take.csi_file)\n        cp=cache/(src.name.replace(\'.csv\',\'\').replace(\'.gz\',\'\')+\'.npz\')\n        # Source and preprocessing version fingerprint; never trust stale caches.\n        key=hashlib.sha256((fingerprint+VERSION+take.csi_file).encode()).hexdigest()\n        if cp.exists():\n            with np.load(cp,allow_pickle=False) as z:\n                cached={k:z[k] for k in z.files}\n            s=cached if str(cached.pop(\'key\'))==key else raw_to_stream(src)\n        else: s=raw_to_stream(src)\n        np.savez(cp,**s,key=key)\n        s.update(pid=take.pid,part=take.part,file=take.csi_file,date=take.date,rx_gain=float(take.rx_gain))\n        lab=pd.read_csv(root/\'labels\'/Path(take.label_file).name).fillna(\'\')\n        assert (lab.pid==take.pid).all()\n        n=len(s[\'t\']); s[\'state\']=np.full(n,-1,np.int64); s[\'fine\']=np.full(n,-1,np.int64)\n        s[\'event\']=np.full(n,-1,np.int64); s[\'events\']=[]\n        ev=lab[lab.type.isin([\'activity\',\'sign\',\'idle\'])].sort_values(\'t_start_us\')\n        last_end=None\n        for ordinal,(_,r) in enumerate(ev.iterrows()):\n            a,b=int(r.t_start_us),int(r.t_end_us)\n            assert b>a,\'Nonpositive label duration\'\n            assert last_end is None or a>=last_end,\'Overlapping ground-truth labels\'\n            last_end=b\n            posture,posture_source=infer_posture(r,take.part)\n            e=dict(id=f\'{take.csi_file}:{ordinal}\',pid=take.pid,part=take.part,file=take.csi_file,\n                   label=r.label,type=r.type,posture=posture,posture_source=posture_source,\n                   trial=r.trial,script=r.script_id,run=r.run,step=r.step,\n                   context=r.context,intent=r.intent,home_action=r.home_action,\n                   t_start_us=a,t_end_us=b,date=take.date,\n                   i0=int(np.searchsorted(s[\'t\'],a)),i1=int(np.searchsorted(s[\'t\'],b)),\n                   covered=a>=s[\'raw_first\'] and b<=s[\'raw_last\'])\n            if not e[\'covered\']:\n                e.update(eligible=False,reason=\'label_outside_recorded_CSI\',quality_fraction=0.)\n                events.append(e); continue\n            i,j=e[\'i0\'],e[\'i1\']; assert j>i\n            typ=STATES.index(r.type); fine=FINE.index(r.label) if r.label in FINE else -1\n            assert r.type==\'idle\' or fine>=0,f\'Unknown fine label {r.label}\'\n            quality=float(s[\'valid\'][i:j].mean())\n            e.update(eligible=True,reason=\'\',quality_fraction=quality,\n                     severe_gap=bool(s[\'severe\'][i:j].any()))\n            s[\'state\'][i:j]=typ; s[\'fine\'][i:j]=fine; s[\'event\'][i:j]=len(s[\'events\'])\n            s[\'events\'].append(e); events.append(e)\n        # Protocol-defined short inter-trial gaps; no inferred labels in long breaks or stories.\n        if take.part!=\'scripts\':\n            es=s[\'events\']\n            for a,b in zip(es[:-1],es[1:]):\n                i,j=a[\'i1\'],b[\'i0\']\n                if 0<(j-i)/FS<=7:\n                    s[\'state\'][i:j]=1 if take.part in [\'actions\',\'actions_seated\'] else 0\n        s[\'state\'][~s[\'valid\']]=-1; s[\'fine\'][~s[\'valid\']]=-1\n        audit.append(dict(pid=take.pid,part=take.part,rows=int(s[\'rows\']),rate=float(s[\'raw_rate\']),\n                          max_gap_ms=float(s[\'max_gap_ms\']),events=len(s[\'events\']),\n                          usable_fraction=float(s[\'valid\'].mean())))\n        streams.append(s)\n        print(\'Prepared\',take.pid,take.part,round(float(s[\'raw_rate\']),1),\'rows/s\',flush=True)\n    pd.DataFrame(audit).to_csv(work/\'recording_audit.csv\',index=False)\n    pd.DataFrame(events).to_csv(work/\'event_coverage.csv\',index=False)\n    atomic_json(dict(version=VERSION,dataset=fingerprint,fs=FS,classes=FINE,\n                     empty_references=\'QC only, excluded from training to avoid shared-day leakage\'),work/\'dataset_info.json\')\n    return streams,fingerprint\n\n\ndef fit_scale(streams):\n    # Equal stream contribution; only training participants enter this function.\n    stats=[]\n    for s in streams:\n        v=s[\'x\'][s[\'valid\'] & (s[\'state\']>=0)][::5]\n        if len(v): stats.append(np.mean(v.astype(np.float64)**2,axis=0))\n    assert stats\n    return np.maximum(np.sqrt(np.mean(stats,axis=0)),1e-4).astype(np.float32)\n\n\ndef window_ok(s,c):\n    a,b=c-MAX_WIN//2,c+MAX_WIN//2\n    return a>=0 and b<=len(s[\'x\']) and not s[\'severe\'][a:b].any() and s[\'valid\'][a:b].mean()>=0.95\n\n\ndef examples(streams):\n    rows=[]\n    for si,s in enumerate(streams):\n        for c in range(MAX_WIN//2,len(s[\'x\'])-MAX_WIN//2+1,HOP):\n            if not window_ok(s,c): continue\n            a,b=c-FS,c+FS\n            v=s[\'state\'][a:b]; vals=np.bincount(v[v>=0],minlength=3)\n            if vals.max()<0.8*(b-a): continue\n            st=int(vals.argmax()); fv=s[\'fine\'][a:b]\n            fine=-1\n            if st!=0:\n                good=fv[fv>=0]\n                if len(good)>=0.8*(b-a):\n                    counts=np.bincount(good,minlength=len(FINE))\n                    if counts.max()>=0.8*(b-a): fine=int(counts.argmax())\n            rows.append((si,c,st,fine))\n    assert rows,\'No valid training windows\'\n    return np.asarray(rows,np.int64)\n\n\nclass WindowDataset(torch.utils.data.Dataset):\n    def __init__(self,streams,rows,scale,seconds):\n        self.streams,self.rows,self.scale,self.width=streams,rows,scale,int(seconds*FS)\n    def __len__(self): return len(self.rows)\n    def __getitem__(self,i):\n        si,c,st,fi=self.rows[i]; half=self.width//2\n        x=(self.streams[si][\'x\'][c-half:c+half]/self.scale).T.copy()\n        return torch.from_numpy(np.clip(x,-20,20)),int(st),int(fi)\n\n\ndef sampling_weights(streams,rows):\n    # Equal states; then equal fine classes; then equal participants within each class.\n    keys=[(int(st),int(fi),streams[si][\'pid\']) for si,c,st,fi in rows]\n    counts=pd.Series(keys).value_counts().to_dict()\n    classes={st:set(fi for a,fi,p in keys if a==st) for st in range(3)}\n    people={(st,fi):set(p for a,b,p in keys if (a,b)==(st,fi)) for st,fi,p in set(keys)}\n    return torch.tensor([1/(counts[k]*len(classes[k[0]])*len(people[k[:2]])) for k in keys],dtype=torch.double)\n\n\nclass ResBlock(nn.Module):\n    def __init__(self,cin,cout,dilation):\n        super().__init__()\n        self.body=nn.Sequential(nn.Conv1d(cin,cout,7,padding=3*dilation,dilation=dilation),\n            nn.BatchNorm1d(cout),nn.GELU(),nn.Conv1d(cout,cout,7,padding=3*dilation,dilation=dilation),nn.BatchNorm1d(cout))\n        self.skip=nn.Conv1d(cin,cout,1) if cin!=cout else nn.Identity()\n    def forward(self,x): return F.gelu(self.body(x)+self.skip(x))\n\n\nclass MultiBlock(nn.Module):\n    def __init__(self,cin,cout=96):\n        super().__init__()\n        self.reduce=nn.Conv1d(cin,24,1,bias=False)\n        self.branches=nn.ModuleList([nn.Conv1d(24,24,k,padding=k//2,bias=False) for k in [9,19,39]])\n        self.pool=nn.Sequential(nn.MaxPool1d(3,stride=1,padding=1),nn.Conv1d(cin,24,1,bias=False))\n        self.norm=nn.GroupNorm(8,cout)\n        self.skip=nn.Conv1d(cin,cout,1) if cin!=cout else nn.Identity()\n        self.se=nn.Sequential(nn.AdaptiveAvgPool1d(1),nn.Conv1d(cout,16,1),nn.GELU(),nn.Conv1d(16,cout,1),nn.Sigmoid())\n    def forward(self,x):\n        z=self.reduce(x)\n        h=self.norm(torch.cat([b(z) for b in self.branches]+[self.pool(x)],dim=1))\n        return F.gelu(h*self.se(h)+self.skip(x))\n\n\nclass CSIModel(nn.Module):\n    def __init__(self,arch):\n        super().__init__(); self.arch=dict(arch)\n        if arch[\'kind\']==\'residual\':\n            self.body=nn.Sequential(nn.BatchNorm1d(52),nn.Conv1d(52,32,5,padding=2),nn.BatchNorm1d(32),nn.GELU(),\n                ResBlock(32,32,1),nn.MaxPool1d(2),ResBlock(32,48,2),nn.MaxPool1d(2),\n                ResBlock(48,64,4),nn.MaxPool1d(2),ResBlock(64,64,8)); width=64\n        else:\n            self.body=nn.Sequential(MultiBlock(52),nn.AvgPool1d(2),MultiBlock(96),nn.AvgPool1d(2),MultiBlock(96)); width=96\n        self.drop=nn.Dropout(0.3); self.state=nn.Linear(width*2,3); self.fine=nn.Linear(width*2,len(FINE))\n    def forward(self,x):\n        h=self.body(x); h=self.drop(torch.cat([h.mean(-1),h.amax(-1)],dim=1))\n        return self.state(h),self.fine(h)\n\n\ndef augment(x):\n    b,c,t=x.shape\n    scale=torch.empty(b,1,1,device=x.device).uniform_(0.8,1.25)\n    noise=torch.randn_like(x)*torch.empty(b,1,1,device=x.device).uniform_(0,0.08)\n    keep=(torch.rand(b,c,1,device=x.device)>0.05)\n    return (x*scale+noise)*keep\n\n\ndef require_kaggle_gpu():\n    assert Path(\'/kaggle/working\').is_dir(),\'Training is restricted to Kaggle, as requested.\'\n    assert torch.cuda.is_available(),\'Enable a GPU in Kaggle Settings before running training.\'\n\n\ndef seed_all(seed):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)\n\n\ndef checkpoint_save(payload,path):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_suffix(\'.tmp\')\n    with open(tmp,\'wb\') as f:\n        torch.save(payload,f); f.flush(); os.fsync(f.fileno())\n    if path.exists(): shutil.copy2(path,path.with_suffix(\'.previous.pt\'))\n    os.replace(tmp,path)\n\n\ndef trusted_load(path):\n    # Only use checkpoints generated by this notebook / your own saved outputs.\n    try: return torch.load(path,map_location=\'cpu\',weights_only=False)\n    except Exception:\n        prev=Path(path).with_suffix(\'.previous.pt\')\n        if not prev.exists(): raise\n        print(\'Recovering previous checkpoint:\',prev)\n        return torch.load(prev,map_location=\'cpu\',weights_only=False)\n\n\ndef unwrapped(model): return model.module if isinstance(model,nn.DataParallel) else model\n\n\ndef predict(model,streams,scale,arch,batch=256,device=None):\n    device=device or next(model.parameters()).device\n    model.eval(); result=[]\n    width=int(arch[\'seconds\']*FS); half=width//2\n    for s in streams:\n        centers=np.arange(MAX_WIN//2,len(s[\'x\'])-MAX_WIN//2+1,HOP)\n        valid=np.array([window_ok(s,int(c)) for c in centers])\n        ps=np.full((len(centers),3),np.nan,np.float32)\n        pf=np.full((len(centers),len(FINE)),np.nan,np.float32)\n        good=np.flatnonzero(valid)\n        with torch.inference_mode():\n            for start in range(0,len(good),batch):\n                ix=good[start:start+batch]\n                x=np.stack([(s[\'x\'][c-half:c+half]/scale).T for c in centers[ix]])\n                xt=torch.from_numpy(np.clip(x,-20,20)).to(device,non_blocking=True)\n                with torch.autocast(\'cuda\',dtype=torch.float16,enabled=str(device).startswith(\'cuda\')):\n                    a,b=model(xt)\n                ps[ix]=a.float().softmax(-1).cpu().numpy(); pf[ix]=b.float().softmax(-1).cpu().numpy()\n        result.append(dict(stream=s,centers=centers,ps=ps,pf=pf,valid=valid))\n    return result\n\n\ndef segment_table(preds):\n    rows=[]\n    for p in preds:\n        s=p[\'stream\']; c=p[\'centers\']\n        for e in s[\'events\']:\n            if e[\'type\'] not in [\'activity\',\'sign\']: continue\n            m=(c>=e[\'i0\'])&(c<e[\'i1\'])&p[\'valid\']\n            if m.any():\n                ps=p[\'ps\'][m].mean(0); pf=p[\'pf\'][m].mean(0)\n                start,end=(0,len(ACTIONS)) if e[\'type\']==\'activity\' else (len(ACTIONS),len(FINE))\n                pred=FINE[start+pf[start:end].argmax()]\n                state=int(ps.argmax())\n                # Explicit branch is clearer than relying on true event type for end-to-end identity.\n                if state==1: hier=ACTIONS[pf[:len(ACTIONS)].argmax()]\n                elif state==2: hier=SIGNS[pf[len(ACTIONS):].argmax()]\n                else: hier=\'idle\'\n                action_prob=pf[:len(ACTIONS)]; action_prob=action_prob/(action_prob.sum()+1e-12)\n            else:\n                pred=hier=\'unavailable\'; ps=np.array([np.nan]*3); action_prob=np.zeros(len(ACTIONS))\n            rows.append(dict(**e,pred=pred,cnn_pred=pred,hier_pred=hier,p_sign=float(ps[2]),\n                             pred_state=\'unavailable\' if not m.any() else STATES[int(ps.argmax())],\n                             scored_windows=int(m.sum()),action_prob=action_prob))\n    return pd.DataFrame(rows)\n\n\ndef macro(frame,labels,pred=\'pred\'):\n    if frame.empty: return float(\'nan\')\n    labels=[c for c in labels if c in set(frame.label)]\n    return float(f1_score(frame.label,frame[pred],labels=labels,average=\'macro\',zero_division=0))\n\n\ndef validation_score(preds):\n    d=segment_table(preds)\n    scores=[]\n    for story in [False,True]:\n        for typ,labels in [(\'activity\',ACTIONS),(\'sign\',SIGNS)]:\n            g=d[(d.part.eq(\'scripts\')==story)&d.type.eq(typ)]\n            if len(g): scores.append(macro(g,labels))\n    g=d[d.part.eq(\'scripts\')]\n    if len(g): scores.append(float(f1_score(g.type,g.pred_state,labels=[\'activity\',\'sign\'],average=\'macro\',zero_division=0)))\n    return float(np.mean(scores))\n\n\n\n# -------- Action baseline and held-out selection --------\ndef action_features(streams,scale):\n    feats=[]; ids=[]; labels=[]\n    for s in streams:\n        if s[\'part\'] not in [\'actions\',\'actions_seated\']: continue\n        for e in s[\'events\']:\n            if e[\'type\']!=\'activity\': continue\n            c=(e[\'i0\']+e[\'i1\'])//2\n            if not window_ok(s,c): continue\n            x=(s[\'x\'][c-2*FS:c+2*FS]/scale).T\n            power=x*x; env=power.mean(0).reshape(20,-1).mean(1); env=env/(env.sum()+1e-9)\n            freq,psd=welch(x,fs=FS,nperseg=100,axis=1)\n            bands=np.stack([psd[:,(freq>=a)&(freq<b)].sum(1) for a,b in [(0.3,1),(1,2),(2,4),(4,8),(8,18)]])\n            bands=bands/(bands.sum(0,keepdims=True)+1e-9)\n            f=np.concatenate([np.log(power.mean(1)+1e-9),np.mean(np.abs(np.diff(x,axis=1)),axis=1),\n                np.quantile(x,0.9,axis=1)-np.quantile(x,0.1,axis=1),env,bands.mean(1),bands.std(1)])\n            feats.append(f); ids.append(e[\'id\']); labels.append(e[\'label\'])\n    return np.asarray(feats,np.float32),ids,np.asarray(labels)\n\n\ndef train_tree(train_streams,scale,path):\n    require_kaggle_gpu()  # Also disallow CPU classifier training on the laptop.\n    x,_,y=action_features(train_streams,scale)\n    clf=ExtraTreesClassifier(n_estimators=300,min_samples_leaf=2,max_features=0.7,class_weight=\'balanced\',random_state=17,n_jobs=2)\n    clf.fit(x,y)\n    tmp=Path(path).with_suffix(\'.tmp\'); joblib.dump(clf,tmp); os.replace(tmp,path)\n    return clf\n\n\ndef tree_predictions(clf,streams,scale):\n    x,ids,y=action_features(streams,scale)\n    if not len(x): return {}\n    out=np.zeros((len(x),len(ACTIONS)),np.float32)\n    out[:,[ACTIONS.index(c) for c in clf.classes_]]=clf.predict_proba(x)\n    return dict(zip(ids,out))\n\n\ndef fuse_table(frame,tree_probs,weight):\n    frame=frame.copy()\n    frame[\'tree_pred\']=\'unavailable\'\n    for i,r in frame.iterrows():\n        if r[\'id\'] not in tree_probs: continue\n        pe=tree_probs[r[\'id\']]\n        frame.at[i,\'tree_pred\']=ACTIONS[int(pe.argmax())]\n        if r[\'scored_windows\']:\n            prob=(1-weight)*r.action_prob+weight*pe\n            frame.at[i,\'pred\']=ACTIONS[int(prob.argmax())]\n    return frame\n\n\nDEFAULT_INTENT={\'HELLO\':(\'I4\',\'wake_system\'),\'THANKYOU\':(\'I4\',\'confirm_close\'),\n \'LAMP\':(\'I1\',\'toggle_light\'),\'FAN\':(\'I1\',\'toggle_fan\'),\'TELEVISION\':(\'I1\',\'toggle_tv\'),\n \'COOK\':(\'I1\',\'cooking_mode\'),\'WATER\':(\'I2\',\'bring_water_reminder\'),\'HOT\':(\'I2\',\'cool_room\'),\n \'HUNGRY\':(\'I2\',\'food_request\'),\'TOILET\':(\'I2\',\'toilet_assist\'),\'CALL\':(\'I3\',\'call_caregiver\'),\n \'HELP\':(\'I3\',\'notify_caregiver\'),\'PAIN\':(\'I3\',\'health_alert\'),\'ACCIDENT\':(\'I3\',\'emergency_alert\')}\nANCHOR={\'eat\':\'meal\',\'drink\':\'meal\',\'chop_food\':\'meal\',\'stir_pot\':\'meal\',\n        \'tvremote\':\'tv\',\'answerphone\':\'tv\',\'wipetable\':\'cleaning\',\'pickup\':\'cleaning\',\'fall\':\'after_fall\'}\n\n\ndef intent(sign,context):\n    if sign==\'HOT\' and context==\'meal\': return (\'I3\',\'kitchen_hazard_alert\')\n    if sign==\'HELP\' and context==\'after_fall\': return (\'I3\',\'emergency_alert\')\n    return DEFAULT_INTENT.get(sign,(\'unknown\',\'unknown\'))\n\n\ndef true_runs(mask):\n    edges=np.diff(np.r_[False,mask,False].astype(int))\n    return list(zip(np.flatnonzero(edges==1),np.flatnonzero(edges==-1)))\n\n\ndef commands(p,tau):\n    # Offline centered windows; 1-second trailing probability average.\n    ps=p[\'ps\']; valid=p[\'valid\']; n=len(ps)\n    raw=np.nan_to_num(ps[:,2]); kernel=np.ones(5)\n    den=np.convolve(valid.astype(float),kernel,\'full\')[:n]\n    sm=np.convolve(raw,kernel,\'full\')[:n]/np.maximum(den,1)\n    mask=(sm>=tau)&valid&(den>=5)\n    # Consecutive signs in a story can have no idle gap. Split only when a\n    # different predicted identity persists for at least one second.\n    identities=np.nan_to_num(p[\'pf\'][:,len(ACTIONS):]).argmax(1)\n    intervals=[]\n    for a,b in true_runs(mask):\n        start=a; active=int(identities[a]); pending=None; since=a\n        for j in range(a+1,b):\n            label=int(identities[j])\n            if label==active: pending=None\n            elif label!=pending: pending=label; since=j\n            elif j-since+1>=5:\n                intervals.append((start,since)); start=since; active=label; pending=None\n        intervals.append((start,b))\n    out=[]\n    for a,b in intervals:\n        if (b-a)*HOP/FS<1.: continue\n        prob=np.nanmean(p[\'pf\'][a:b,len(ACTIONS):],axis=0)\n        label=SIGNS[int(prob.argmax())]\n        i0=int(p[\'centers\'][a]-HOP//2); i1=int(p[\'centers\'][b-1]+HOP//2)\n        # Context uses only predicted prior activity frames, never true context.\n        recent=(p[\'centers\']<i0)&(p[\'centers\']>=i0-120*FS)&valid\n        good=np.flatnonzero(recent & (np.nan_to_num(ps[:,1])>=0.6))\n        ctx=\'isolated\'\n        for j in good:\n            action=ACTIONS[int(p[\'pf\'][j,:len(ACTIONS)].argmax())]\n            if action in ANCHOR: ctx=ANCHOR[action]\n        cat,home=intent(label,ctx)\n        out.append(dict(i0=i0,i1=i1,label=label,context=ctx,intent=cat,home_action=home))\n    return out\n\n\ndef event_metrics(preds,tau):\n    totals=dict(n_true=0,n_pred=0,detected=0,identified=0,intent_correct=0,false_on_routine=0,routine_seconds=0.)\n    allcommands=[]\n    for p in preds:\n        s=p[\'stream\']\n        if s[\'part\']!=\'scripts\': continue\n        gt=[e for e in s[\'events\'] if e[\'type\']==\'sign\']\n        cmds=commands(p,tau); totals[\'n_true\']+=len(gt); totals[\'n_pred\']+=len(cmds)\n        # Match detection without using sign identity, one-to-one, highest IoU first.\n        pairs=[]\n        for i,c in enumerate(cmds):\n            for j,e in enumerate(gt):\n                inter=max(0,min(c[\'i1\'],e[\'i1\'])-max(c[\'i0\'],e[\'i0\']))\n                union=max(c[\'i1\'],e[\'i1\'])-min(c[\'i0\'],e[\'i0\'])\n                if inter and inter/union>=0.10: pairs.append((inter/union,i,j))\n        used_c=set(); used_e=set()\n        for _,i,j in sorted(pairs,reverse=True):\n            if i in used_c or j in used_e: continue\n            used_c.add(i); used_e.add(j); totals[\'detected\']+=1\n            correct=cmds[i][\'label\']==gt[j][\'label\']; totals[\'identified\']+=int(correct)\n            totals[\'intent_correct\']+=int(correct and cmds[i][\'intent\']==gt[j][\'intent\'] and cmds[i][\'home_action\']==gt[j][\'home_action\'])\n        # Denominator only measured routine time with model coverage, not all story wall time.\n        covered=np.zeros(len(s[\'state\']),bool)\n        for c,v in zip(p[\'centers\'],p[\'valid\']):\n            if v: covered[max(0,c-HOP//2):min(len(covered),c+HOP//2)]=True\n        routine=(s[\'state\']==1)&covered\n        totals[\'routine_seconds\']+=float(routine.sum()/FS)\n        for i,c in enumerate(cmds):\n            midpoint=(c[\'i0\']+c[\'i1\'])//2\n            if i not in used_c and 0<=midpoint<len(routine) and routine[midpoint]: totals[\'false_on_routine\']+=1\n            allcommands.append(dict(pid=s[\'pid\'],file=s[\'file\'],matched=i in used_c,**c))\n    for key,count in [(\'detection\',totals[\'detected\']),(\'identity\',totals[\'identified\'])]:\n        precision=count/max(totals[\'n_pred\'],1); recall=count/max(totals[\'n_true\'],1)\n        totals[key+\'_precision\']=precision; totals[key+\'_recall\']=recall\n        totals[key+\'_f1\']=2*precision*recall/max(precision+recall,1e-12)\n    totals[\'intent_e2e\']=totals[\'intent_correct\']/max(totals[\'n_true\'],1)\n    totals[\'false_commands_per_routine_hour\']=totals[\'false_on_routine\']/max(totals[\'routine_seconds\']/3600,1e-12)\n    totals[\'tau\']=float(tau)\n    return totals,allcommands\n\n\ndef choose_tau(inner_predictions):\n    rows=[]\n    for tau in np.arange(0.2,0.901,0.05):\n        rr=[event_metrics(p,float(tau))[0] for p in inner_predictions]\n        rows.append(dict(tau=float(tau),identity_f1=float(np.mean([r[\'identity_f1\'] for r in rr])),\n                         detection_f1=float(np.mean([r[\'detection_f1\'] for r in rr]))))\n    # Identity first; detection breaks ties; then prefer a higher threshold.\n    selected=max(rows,key=lambda r:(r[\'identity_f1\'],r[\'detection_f1\'],r[\'tau\']))\n    return selected[\'tau\'],rows\n\n\ndef average_predictions(all_preds):\n    result=[]\n    for pos in range(len(all_preds[0])):\n        p=all_preds[0][pos].copy()\n        # NaNs at invalid frames are intentional; average only valid frames.\n        for key in [\'ps\',\'pf\']:\n            p[key]=np.stack([a[pos][key] for a in all_preds]).mean(0)\n        result.append(p)\n    return result\n\n\ndef save_predictions(preds,path):\n    arrays={}\n    for i,p in enumerate(preds):\n        for key in [\'centers\',\'ps\',\'pf\',\'valid\']: arrays[f\'{i}_{key}\']=p[key]\n        arrays[f\'{i}_file\']=np.array(p[\'stream\'][\'file\'])\n    np.savez_compressed(path,**arrays)\n\n\ndef release(net):\n    net.cpu(); torch.cuda.empty_cache(); gc.collect()\n\n\ndef report_fold(table,preds,tau,folder):\n    folder=Path(folder); folder.mkdir(parents=True,exist_ok=True)\n    table.drop(columns=[\'action_prob\']).to_csv(folder/\'segments.csv\',index=False)\n    metrics=[]; perclass=[]\n    # Common 10 actions /14 signs; expanded results show actual support, never fictitious zeros.\n    slices=[(\'isolated\',table[table.part.ne(\'scripts\')]),(\'stories_oracle_boundaries\',table[table.part.eq(\'scripts\')]),\n            (\'isolated_seated\',table[table.part.isin([\'actions_seated\',\'signs_seated\'])]),\n            (\'isolated_original_parts\',table[table.part.isin([\'actions\',\'signs_standing\'])])]\n    for split,d in slices:\n        for typ,names in [(\'activity\',ACTIONS),(\'sign\',SIGNS)]:\n            g=d[d.type.eq(typ)]\n            if g.empty: continue\n            supported=[n for n in names if n in set(g.label)]\n            values=dict(split=split,type=typ,n=len(g),supported_classes=len(supported),\n                        macro_f1=macro(g,supported),accuracy=float((g.label==g.pred).mean()),\n                        cnn_macro_f1=macro(g,supported,\'cnn_pred\'),\n                        unavailable=int(g.scored_windows.eq(0).sum()))\n            if typ==\'activity\': values[\'common10_macro_f1\']=macro(g[g.label.isin(ACTIONS[:10])],ACTIONS[:10])\n            metrics.append(values)\n            rep=classification_report(g.label,g.pred,labels=supported,output_dict=True,zero_division=0)\n            for label in supported: perclass.append(dict(split=split,type=typ,label=label,**rep[label]))\n    ev,cmd=event_metrics(preds,tau)\n    atomic_json(ev,folder/\'story_events.json\'); pd.DataFrame(cmd).to_csv(folder/\'predicted_commands.csv\',index=False)\n    pd.DataFrame(metrics).to_csv(folder/\'metrics.csv\',index=False)\n    pd.DataFrame(perclass).to_csv(folder/\'per_class.csv\',index=False)\n    st=table[table.part.eq(\'scripts\')]; good=st.p_sign.notna()\n    atomic_json(dict(story_auc=roc_auc_score(st.loc[good,\'type\'].eq(\'sign\'),st.loc[good,\'p_sign\']) if st.loc[good,\'type\'].nunique()==2 else None,\n                     story_hierarchical_accuracy=float(st.label.eq(st.hier_pred).mean()),\n                     story_oracle_boundary_note=\'Fine identity uses known event boundaries/type; event detection and intent metrics do not.\',\n                     missing_prediction_policy=\'Fully covered labels without usable windows remain as unavailable and count as misses.\'),folder/\'context.json\')\n\n\n\n\n\n\n\n\ndef archive_outputs(out,only_fold=None):\n    out=Path(out); base=out.parent\n    name=f\'FYP_{only_fold}_checkpoint_bundle\' if only_fold else \'FYP_all_checkpoints_and_results\'\n    dest=base/(name+\'.zip\'); tmp=dest.with_suffix(\'.zip.part\')\n    with zipfile.ZipFile(tmp,\'w\',zipfile.ZIP_DEFLATED,compresslevel=1) as z:\n        for p in out.rglob(\'*\'):\n            if not p.is_file() or \'cache\' in p.parts or p.suffix in [\'.tmp\',\'.part\']: continue\n            rel=p.relative_to(out)\n            if only_fold and len(rel.parts)>1 and rel.parts[0] not in [only_fold]: continue\n            if only_fold and len(rel.parts)==1 and rel.name not in [\'fyp_pipeline.py\',\'dataset_info.json\',\'environment.json\',\'recording_audit.csv\',\'event_coverage.csv\']: continue\n            z.write(p,Path(\'results\')/rel)\n    tmp.replace(dest)\n    atomic_json(dict(sha256=sha256(dest),bytes=dest.stat().st_size),dest.with_suffix(\'.zip.json\'))\n    return dest\n\n\n\n\ndef summarize(out):\n    import matplotlib.pyplot as plt\n    out=Path(out); frames=[]; metrics=[]; events=[]\n    for pid in PIDS:\n        if not (out/pid/\'complete.json\').exists(): continue\n        d=pd.read_csv(out/pid/\'segments.csv\'); frames.append(d)\n        m=pd.read_csv(out/pid/\'metrics.csv\'); m[\'pid\']=pid; metrics.append(m)\n        ev=json.loads((out/pid/\'story_events.json\').read_text()); ev[\'pid\']=pid; events.append(ev)\n    if not frames: print(\'No completed folds yet.\'); return\n    d=pd.concat(frames,ignore_index=True); m=pd.concat(metrics,ignore_index=True); ev=pd.DataFrame(events)\n    d.to_csv(out/\'all_out_of_fold_predictions.csv\',index=False); m.to_csv(out/\'fold_metrics.csv\',index=False); ev.to_csv(out/\'story_metrics.csv\',index=False)\n    figures=out/\'figures\'; figures.mkdir(exist_ok=True)\n    fig,ax=plt.subplots(figsize=(10,5))\n    for pid in PIDS:\n        for path in sorted((out/pid/\'models\').rglob(\'history.csv\')):\n            h=pd.read_csv(path)\n            ax.plot(h.epoch,h.loss,label=f\'{pid} {path.parent.parent.name}/{path.parent.name}\',alpha=0.7)\n    ax.set_title(\'Final fold training losses\'); ax.set_xlabel(\'Epoch\'); ax.set_ylabel(\'Loss\')\n    ax.legend(fontsize=7,ncol=3); fig.tight_layout(); fig.savefig(figures/\'training_curves.png\',dpi=160); plt.show(); plt.close(fig)\n    for typ,names in [(\'activity\',ACTIONS),(\'sign\',SIGNS)]:\n        for source,mask in [(\'isolated\',d.part.ne(\'scripts\')),(\'stories_oracle\',d.part.eq(\'scripts\'))]:\n            g=d[mask & d.type.eq(typ)]\n            if g.empty: continue\n            report=classification_report(g.label,g.pred,labels=names,output_dict=True,zero_division=0)\n            per=pd.DataFrame([dict(label=n,**report[n]) for n in names]); per.to_csv(out/f\'{source}_{typ}_per_class.csv\',index=False)\n            fig,ax=plt.subplots(figsize=(11,5)); values=per[\'f1-score\'].to_numpy()\n            ax.bar(names,values,color=[\'#187b62\' if v>=0.6 else \'#ca8a32\' for v in values]); ax.set_ylim(0,1)\n            ax.set_ylabel(\'F1\'); ax.set_title(f\'{source.replace("_"," ")} — {typ}; pooled held-out predictions\')\n            ax.tick_params(axis=\'x\',rotation=55); fig.tight_layout(); fig.savefig(figures/f\'{source}_{typ}_f1.png\',dpi=160); plt.show(); plt.close(fig)\n            # Rectangular confusion includes idle / unavailable instead of silently dropping missed examples.\n            pred_labels=names+[\'idle\',\'unavailable\']+[n for n in FINE if n not in names]\n            cm=np.array([[(g.label.eq(a)&g.pred.eq(b)).sum() for b in pred_labels] for a in names])\n            used=cm.sum(0)>0; cm=cm[:,used]; cols=np.array(pred_labels)[used]\n            norm=cm/np.maximum(cm.sum(1,keepdims=True),1)\n            fig,ax=plt.subplots(figsize=(11,8)); im=ax.imshow(norm,vmin=0,vmax=1,cmap=\'Blues\')\n            ax.set_xticks(range(len(cols)),cols,rotation=70); ax.set_yticks(range(len(names)),names)\n            ax.set_xlabel(\'Predicted\'); ax.set_ylabel(\'True\'); ax.set_title(f\'{source} {typ}: row-normalized confusion\')\n            fig.colorbar(im,ax=ax); fig.tight_layout(); fig.savefig(figures/f\'{source}_{typ}_confusion.png\',dpi=160); plt.show(); plt.close(fig)\n    # Participant, posture, common-vs-expanded evidence remain visible.\n    pp=[]\n    for (pid,part,posture,typ),g in d.groupby([\'pid\',\'part\',\'posture\',\'type\'],dropna=False):\n        pp.append(dict(pid=pid,part=part,posture=posture,type=typ,n=len(g),macro_f1=macro(g,ACTIONS if typ==\'activity\' else SIGNS)))\n    pd.DataFrame(pp).to_csv(out/\'posture_participant_metrics.csv\',index=False)\n    summary=m.groupby([\'split\',\'type\']).agg(mean_f1=(\'macro_f1\',\'mean\'),sd_f1=(\'macro_f1\',\'std\'),participants=(\'pid\',\'nunique\'),events=(\'n\',\'sum\')).reset_index()\n    summary.to_csv(out/\'summary.csv\',index=False)\n    fig,ax=plt.subplots(figsize=(9,5))\n    iso=m[m.split.eq(\'isolated\')].pivot(index=\'pid\',columns=\'type\',values=\'macro_f1\'); iso.plot.bar(ax=ax,ylim=(0,1))\n    ax.set_title(\'Unseen-participant performance\'); ax.set_ylabel(\'Macro F1\'); fig.tight_layout(); fig.savefig(figures/\'participant_f1.png\',dpi=160); plt.show(); plt.close(fig)\n    fig,ax=plt.subplots(figsize=(9,5)); ev.set_index(\'pid\')[[\'detection_recall\',\'identity_recall\',\'intent_e2e\']].plot.bar(ax=ax,ylim=(0,1))\n    ax.set_title(\'Stories: detection, correct sign, correct intent\'); fig.tight_layout(); fig.savefig(figures/\'story_end_to_end.png\',dpi=160); plt.show(); plt.close(fig)\n    print(summary.to_string(index=False)); print(ev[[\'pid\',\'n_true\',\'detection_f1\',\'identity_f1\',\'intent_e2e\',\'false_commands_per_routine_hour\']].to_string(index=False))\n    print(\'Completed participants:\',sorted(d.pid.unique()),\'of\',PIDS)\n    if len(frames)<5: print(\'PARTIAL EVALUATION — finish the remaining folds before reporting a five-person result.\')\n    return summary\n\n\ndef load_deployment(folder,device=\'cuda\'):\n    folder=Path(folder); card=json.loads((folder/\'model_card.json\').read_text()); nets=[]; scale=None\n    for path in sorted(folder.glob(\'seed_*/best.pt\')):\n        ck=trusted_load(path); net=CSIModel(ck[\'arch\']).to(device); net.load_state_dict(ck[\'model\']); net.eval()\n        nets.append(net); scale=ck[\'scale\']\n    assert nets,\'No saved model weights\'\n    return nets,scale,card\n\n\ndef predict_unlabelled_csv(csv_path,model_folder,device=\'cuda\'):\n    nets,scale,card=load_deployment(model_folder,device)\n    s=raw_to_stream(csv_path); s.update(file=Path(csv_path).name,pid=\'new\',part=\'inference\',events=[])\n    pred=average_predictions([predict(n,[s],scale,card[\'arch\'],device=device) for n in nets])[0]\n    # Unlabelled continuous inference uses CNNs, not oracle-boundary ExtraTrees fusion.\n    rows=[]\n    for c,ps,pf,valid in zip(pred[\'centers\'],pred[\'ps\'],pred[\'pf\'],pred[\'valid\']):\n        if not valid: label=state=\'unavailable\'; confidence=0.\n        else:\n            state=STATES[int(ps.argmax())]\n            labels=ACTIONS if state==\'activity\' else SIGNS\n            f=pf[:len(ACTIONS)] if state==\'activity\' else pf[len(ACTIONS):]\n            label=\'idle\' if state==\'idle\' else labels[int(f.argmax())]\n            confidence=float(ps.max())\n        rows.append(dict(timestamp_us=int(s[\'t\'][c]),state=state,label=label,state_confidence=confidence))\n    return pd.DataFrame(rows),pd.DataFrame(commands(pred,card[\'tau\']))\n\n# Fast path: no DataLoader workers, no DataParallel. Each process owns one GPU.\ndef gpu_bank(streams,scale):\n    offsets=np.cumsum([0]+[len(s[\'x\']) for s in streams[:-1]])\n    flat=np.concatenate([np.clip(s[\'x\']/scale,-20,20) for s in streams],axis=0).T.copy()\n    bank=torch.from_numpy(flat).cuda()\n    rows=examples(streams)\n    centers=rows[:,1]+offsets[rows[:,0]]\n    return bank,torch.as_tensor(centers,device=\'cuda\'),torch.as_tensor(rows[:,2:],device=\'cuda\'),sampling_weights(streams,rows).float().cuda()\n\n\ndef epoch_steps(n_windows,cfg):\n    return max(32,int(np.ceil(n_windows*cfg[\'passes_per_epoch\']/cfg[\'batch_size\'])))\n\n\ndef fit_fast(tr,va,arch,cfg,folder,fingerprint,epochs=None,deadline=None):\n    global CURRENT_JOB\n    require_kaggle_gpu(); torch.set_num_threads(2); seed_all(cfg[\'seed\'])\n    torch.backends.cudnn.benchmark=True; torch.set_float32_matmul_precision(\'high\')\n    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)\n    CURRENT_JOB=str(folder.relative_to(folder.parents[3])) if len(folder.parents)>3 else str(folder)\n    progress(\'building GPU sample bank\')\n    scale=fit_scale(tr);bank,centers,targets,weights=gpu_bank(tr,scale)\n    cfg=dict(cfg)\n    cfg[\'steps_per_epoch\']=epoch_steps(len(centers),cfg)\n    net=CSIModel(arch).cuda()\n    optimizer=torch.optim.AdamW(net.parameters(),lr=cfg[\'lr\'],weight_decay=cfg[\'weight_decay\'])\n    total=int(epochs or cfg[\'epochs\'])\n    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=cfg[\'epochs\'],eta_min=1e-5)\n    scaler=torch.amp.GradScaler(\'cuda\'); width=int(arch[\'seconds\']*FS)\n    times=torch.arange(width,device=\'cuda\')-width//2\n    best=-1.;best_epoch=1;best_weights=None;bad=0;history=[]\n    print(f\'[JOB] {folder}: {total} epochs maximum, {cfg["steps_per_epoch"]} batches/epoch\',flush=True)\n    for epoch in range(total):\n        if deadline is not None and time.time()>=deadline:\n            if best_weights is None: raise SessionBudgetReached(\'No epoch completed before time budget.\')\n            break\n        started=time.time();net.train();losses=[];progress(f\'epoch {epoch+1} training\')\n        picks=torch.multinomial(weights,cfg[\'steps_per_epoch\']*cfg[\'batch_size\'],replacement=True).reshape(cfg[\'steps_per_epoch\'],cfg[\'batch_size\'])\n        for step,ix in enumerate(picks):\n            if step%8==0:\n                progress(f\'epoch {epoch+1}, batch {step+1}/{cfg["steps_per_epoch"]}\')\n                if deadline is not None and time.time()>=deadline:\n                    raise SessionBudgetReached(\'GPU training budget reached between batches.\')\n            idx=centers[ix,None]+times[None,:]\n            x=bank[:,idx].permute(1,0,2).contiguous();ys=targets[ix,0];yf=targets[ix,1]\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(\'cuda\',dtype=torch.float16):\n                st,fi=net(augment(x));loss=F.cross_entropy(st,ys,label_smoothing=0.03)\n                has=yf>=0\n                if has.any():loss=loss+F.cross_entropy(fi[has],yf[has],label_smoothing=0.05)\n            if not torch.isfinite(loss):raise RuntimeError(\'Nonfinite loss; previous checkpoint is retained.\')\n            scaler.scale(loss).backward();scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(net.parameters(),1.)\n            scaler.step(optimizer);scaler.update();losses.append(float(loss.detach()))\n        scheduler.step()\n        # Validate at a fixed cadence. Keep the same cosine horizon for selection and refit.\n        score=None;probe={}\n        if va and ((epoch+1)%cfg[\'val_every\']==0 or epoch+1==total):\n            progress(f\'epoch {epoch+1} validation\')\n            vp=predict(net,va,scale,arch,cfg[\'batch_size\']);score=validation_score(vp)\n            probe=training_probe(net,bank,centers,targets,width,cfg[\'batch_size\'])\n        improved=(not va) or (score is not None and score>best+1e-4)\n        if improved:\n            best=score if score is not None else 0.;best_epoch=epoch+1;bad=0\n            best_weights={k:v.detach().cpu().clone() for k,v in net.state_dict().items()}\n        elif score is not None:bad+=1\n        history.append(dict(epoch=epoch+1,loss=float(np.mean(losses)),validation_score=score,seconds=time.time()-started,\n            steps=cfg[\'steps_per_epoch\'],draws=cfg[\'steps_per_epoch\']*cfg[\'batch_size\'],learning_rate=optimizer.param_groups[0][\'lr\'],**probe))\n        payload=dict(version=VERSION,arch=arch,classes=FINE,states=STATES,scale=scale,dataset=fingerprint,\n            epoch=epoch+1,config=cfg,model={k:v.detach().cpu() for k,v in net.state_dict().items()},optimizer=optimizer.state_dict(),\n            rng_python=random.getstate(),rng_numpy=np.random.get_state(),rng_torch=torch.get_rng_state(),rng_cuda=torch.cuda.get_rng_state_all(),\n            scheduler=scheduler.state_dict(),scaler=scaler.state_dict(),train_participants=sorted({s[\'pid\'] for s in tr}),\n            validation_participants=sorted({s[\'pid\'] for s in va}),history=history)\n        progress(f\'epoch {epoch+1} saving\')\n        checkpoint_save(payload,folder/\'last.pt\')\n        if improved:\n            checkpoint_save(dict(model=best_weights,arch=arch,classes=FINE,states=STATES,scale=scale,dataset=fingerprint,epoch=best_epoch),folder/\'best.pt\')\n        pd.DataFrame(history).to_csv(folder/\'history.csv\',index=False)\n        print(f\'[EPOCH] {folder.name}/{arch["name"]} {epoch+1}/{total} loss={np.mean(losses):.4f} val={score} elapsed={time.time()-started:.1f}s\',flush=True)\n        progress(f\'epoch {epoch+1} completed\')\n        if va and epoch+1>=cfg[\'min_epochs\'] and bad>=cfg[\'patience\']:break\n    if best_weights is None:raise SessionBudgetReached(\'No validated model completed within the budget.\')\n    net.load_state_dict(best_weights)\n    del bank,centers,targets,weights,optimizer,scheduler,scaler,picks\n    torch.cuda.empty_cache()\n    return net,scale,best_epoch,history\n\n\n\n\n\n\ndef fast_worker(spec_path):\n    import joblib\n    global PROGRESS_PATH,CURRENT_JOB\n    spec=json.loads(Path(spec_path).read_text());PROGRESS_PATH=Path(spec[\'progress\']);CURRENT_JOB=spec[\'pid\']\n    torch.set_num_threads(2)\n    require_kaggle_gpu();progress(\'loading preprocessed data\')\n    streams=joblib.load(spec[\'streams\'])\n    try:\n        if spec[\'pid\']==\'FINAL\':fast_final(streams,spec[\'cfg\'],spec[\'out\'],spec[\'fingerprint\'],spec[\'deadline\'])\n        else:fast_fold(streams,spec[\'pid\'],spec[\'cfg\'],spec[\'out\'],spec[\'fingerprint\'],spec[\'deadline\'])\n    except SessionBudgetReached as exc:\n        print(\'[TIME BUDGET]\',exc,flush=True);progress(\'time budget reached\');return 75\n    return 0\n\n\n# Validation-tested alternative: local waveform shape plus retained energy.\n_BaseCSIModel=CSIModel\nclass CSIModel(_BaseCSIModel):\n    def __init__(self,arch):\n        super().__init__(arch)\n        self.shape_input=arch.get(\'representation\')==\'window_normalized\'\n        if self.shape_input:\n            def replace_norm(module):\n                for name,child in list(module.named_children()):\n                    if isinstance(child,nn.BatchNorm1d):\n                        groups=8 if child.num_features%8==0 else 4\n                        setattr(module,name,nn.GroupNorm(groups,child.num_features))\n                    else:replace_norm(child)\n            replace_norm(self.body)\n            self.state=nn.Linear(self.state.in_features+2,3)\n            self.fine=nn.Linear(self.fine.in_features+2,len(FINE))\n    def forward(self,x):\n        if not self.shape_input:return super().forward(x)\n        energy=x.square().mean((1,2)).clamp_min(1e-4)\n        velocity=(x[:,:,1:]-x[:,:,:-1]).square().mean((1,2)).clamp_min(1e-4)\n        shape=x/energy.sqrt()[:,None,None]\n        h=self.body(shape.clamp(-20,20));h=torch.cat([h.mean(-1),h.amax(-1),energy.log()[:,None],velocity.log()[:,None]],1)\n        h=self.drop(h);return self.state(h),self.fine(h)\n\n\ndef augment(x):\n    b,c,t=x.shape\n    # Noise is relative to each window, so weak seated movements are not buried\n    # beneath the same absolute noise amplitude used for large walking motions.\n    rms=x.square().mean((1,2),keepdim=True).sqrt().clamp_min(1e-3)\n    gain=torch.empty(b,1,1,device=x.device).uniform_(0.65,1.5)\n    sc_gain=torch.empty(b,c,1,device=x.device).uniform_(0.85,1.15)\n    polarity=torch.where(torch.rand(b,1,1,device=x.device)<0.5,-1.,1.)\n    noise=torch.randn_like(x)*rms*torch.empty(b,1,1,device=x.device).uniform_(0,0.04)\n    keep=(torch.rand(b,c,1,device=x.device)>0.05)\n    return (x*gain*sc_gain*polarity+noise)*keep\n\n\ndef training_probe(net,bank,centers,targets,width,batch):\n    net.eval()\n    with torch.inference_mode():\n        ix=torch.linspace(0,len(centers)-1,min(batch,len(centers)),device=centers.device).long()\n        times=torch.arange(width,device=centers.device)-width//2\n        x=bank[:,centers[ix,None]+times].permute(1,0,2).contiguous()\n        with torch.autocast(\'cuda\',dtype=torch.float16):st,fi=net(x)\n        ys,yf=targets[ix,0],targets[ix,1]\n        result=dict(probe_state_accuracy=float((st.argmax(1)==ys).float().mean()))\n        for name,state,a,b in [(\'action\',1,0,12),(\'sign\',2,12,26)]:\n            m=(ys==state)&(yf>=0)\n            result[\'probe_\'+name+\'_typed_accuracy\']=float((a+fi[m,a:b].argmax(1)==yf[m]).float().mean()) if m.any() else None\n    return result\n\n\ndef role_scores(predictions):\n    d=segment_table(predictions);out={}\n    for role,typ,names in [(\'action\',\'activity\',ACTIONS),(\'sign\',\'sign\',SIGNS)]:\n        scores=[]\n        for story in [False,True]:\n            g=d[d.type.eq(typ)&(d.part.eq(\'scripts\')==story)]\n            if len(g):scores.append(macro(g,names))\n        out[role]=float(np.mean(scores))\n    story=d[d.part.eq(\'scripts\')]\n    out[\'detector\']=float(f1_score(story.type,story.pred_state,labels=[\'activity\',\'sign\'],average=\'macro\',zero_division=0))\n    return out\n\n\ndef combine_roles(by_arch,roles):\n    detector=by_arch[roles[\'detector\']];action=by_arch[roles[\'action\']];sign=by_arch[roles[\'sign\']]\n    combined=[]\n    for a,b,c in zip(detector,action,sign):\n        assert np.array_equal(a[\'centers\'],b[\'centers\']) and np.array_equal(a[\'centers\'],c[\'centers\'])\n        p=a.copy();p[\'pf\']=a[\'pf\'].copy()\n        pa=b[\'pf\'][:,:len(ACTIONS)];ps=c[\'pf\'][:,len(ACTIONS):]\n        p[\'pf\'][:,:len(ACTIONS)]=pa/(pa.sum(1,keepdims=True)+1e-12)\n        p[\'pf\'][:,len(ACTIONS):]=ps/(ps.sum(1,keepdims=True)+1e-12)\n        p[\'valid\']=a[\'valid\']&b[\'valid\']&c[\'valid\'];combined.append(p)\n    return combined\n\n\ndef fast_fold(streams,test,cfg,out,fingerprint,deadline):\n    require_kaggle_gpu();out=Path(out);fold=out/test;fold.mkdir(parents=True,exist_ok=True)\n    val=VALIDATION_PID[test]\n    tr=[s for s in streams if s[\'pid\'] not in [test,val]];va=[s for s in streams if s[\'pid\']==val]\n    refit=[s for s in streams if s[\'pid\']!=test];te=[s for s in streams if s[\'pid\']==test]\n    assert not {test,val}&{s[\'pid\'] for s in tr}\n    rows=[];vp={};epochs={};architectures={a[\'name\']:a for a in CANDIDATES}\n    for arch in CANDIDATES:\n        net,scale,ep,_=fit_fast(tr,va,arch,cfg,fold/\'selection\'/arch[\'name\'],fingerprint,deadline=deadline)\n        progress(\'role-specific validation selection\')\n        pred=predict(net,va,scale,arch,cfg[\'batch_size\']);scores=role_scores(pred)\n        rows.append(dict(candidate=arch[\'name\'],validation_pid=val,best_epoch=ep,**scores))\n        vp[arch[\'name\']]=pred;epochs[arch[\'name\']]=ep;release(net);del net\n    roles={r:max(rows,key=lambda a:a[r])[\'candidate\'] for r in [\'detector\',\'action\',\'sign\']}\n    pd.DataFrame(rows).to_csv(fold/\'validation_selection.csv\',index=False)\n    combined=combine_roles(vp,roles);tau,tau_rows=choose_tau([combined]);vd=segment_table(combined)\n    pd.DataFrame(tau_rows).to_csv(fold/\'validation_thresholds.csv\',index=False)\n    progress(\'validation ExtraTrees comparison\')\n    tree_scale=fit_scale(tr);tree=train_tree(tr,tree_scale,fold/\'selection_action_tree.joblib\');tp=tree_predictions(tree,va,tree_scale)\n    fusion=[]\n    for weight in [0.,0.5,1.]:\n        g=fuse_table(vd,tp,weight);g=g[g.type.eq(\'activity\')&g.part.ne(\'scripts\')]\n        fusion.append(dict(weight=weight,score=macro(g,ACTIONS)))\n    weight=max(fusion,key=lambda r:r[\'score\'])[\'weight\']\n    choice=dict(roles=roles,architectures=architectures,epochs=epochs,tau=tau,action_tree_weight=weight,\n        validation_pid=val,training_pids=[p for p in PIDS if p!=test],test_pid=test,validation_fusion=fusion,\n        protocol=\'Fixed participant validation; role-specific architecture; two-seed refit; dataset-scaled epochs.\')\n    atomic_json(choice,fold/\'selection.json\')\n    pred_by_arch={}\n    for name in sorted(set(roles.values())):\n        ens=[]\n        for seed in cfg[\'seeds\']:\n            actual=dict(cfg,seed=seed)\n            net,scale,_,_=fit_fast(refit,[],architectures[name],actual,fold/\'models\'/name/f\'seed_{seed}\',fingerprint,epochs=epochs[name],deadline=deadline)\n            progress(\'held-out inference \'+name+\' seed \'+str(seed))\n            ens.append(predict(net,te,scale,architectures[name],cfg[\'batch_size\']));release(net);del net\n        pred_by_arch[name]=average_predictions(ens)\n    pred=combine_roles(pred_by_arch,roles);d=segment_table(pred)\n    tree_scale=fit_scale(refit);tree=train_tree(refit,tree_scale,fold/\'action_tree.joblib\')\n    d=fuse_table(d,tree_predictions(tree,te,tree_scale),weight)\n    save_predictions(pred,fold/\'frame_predictions.npz\');report_fold(d,pred,tau,fold)\n    atomic_json(dict(test=test,version=VERSION,protocol=choice[\'protocol\']),fold/\'complete.json\')\n    progress(\'fold completed\');print(\'[FOLD COMPLETE]\',test,roles,flush=True)\n\n\ndef fast_final(streams,cfg,out,fingerprint,deadline):\n    require_kaggle_gpu();out=Path(out);completed=[p for p in PIDS if (out/p/\'complete.json\').exists()]\n    if not completed:raise SessionBudgetReached(\'No complete validation/evaluation fold; refusing an unselected deployment refit.\')\n    choices=[json.loads((out/p/\'selection.json\').read_text()) for p in completed]\n    names=[a[\'name\'] for a in CANDIDATES];architectures={a[\'name\']:a for a in CANDIDATES}\n    roles={r:max(names,key=lambda n:sum(c[\'roles\'][r]==n for c in choices)) for r in [\'detector\',\'action\',\'sign\']}\n    final=out/\'deployment_all5\';final.mkdir(exist_ok=True);saved={};durations={}\n    for name in sorted(set(roles.values())):\n        epochs=max(8,int(np.median([c[\'epochs\'][name] for c in choices])))\n        durations[name]=epochs;saved[name]=[]\n        for seed in cfg[\'seeds\']:\n            path=final/\'models\'/name/f\'seed_{seed}\'\n            net,scale,_,_=fit_fast(streams,[],architectures[name],dict(cfg,seed=seed),path,fingerprint,epochs=epochs,deadline=deadline)\n            release(net);del net;saved[name].append(str((path/\'best.pt\').relative_to(final)))\n    progress(\'all-five action tree\')\n    train_tree(streams,fit_scale(streams),final/\'action_tree.joblib\')\n    atomic_json(dict(roles=roles,architectures=architectures,epochs=durations,models=saved,seeds=cfg[\'seeds\'],\n        tau=float(np.median([c[\'tau\'] for c in choices])),action_tree_weight=float(pd.Series([c[\'action_tree_weight\'] for c in choices]).mode().iloc[0]),\n        classes=FINE,states=STATES,fs=FS,training_participants=PIDS,dataset=fingerprint,completed_evaluation_folds=completed,\n        evaluation_status=\'complete\' if len(completed)==5 else \'PARTIAL\',\n        validation=\'Deployment refit has no independent score. Changes were informed by the earlier benchmark; fresh recordings are needed for final confirmation.\',\n        inference=\'Offline centered windows; role-specific two-seed ensemble. Tree fusion is isolated-event only.\'),final/\'model_card.json\')\n    shutil.copy2(__file__,final/\'fyp_fast.py\');progress(\'deployment refit completed\')\n    print(\'[ALL-FIVE MODEL SAVED]\',final,flush=True)\n\n\ndef load_deployment(folder,device=\'cuda\'):\n    folder=Path(folder);card=json.loads((folder/\'model_card.json\').read_text());models={}\n    for name,paths in card[\'models\'].items():\n        models[name]=[]\n        for path in paths:\n            ck=trusted_load(folder/path);net=CSIModel(ck[\'arch\']).to(device);net.load_state_dict(ck[\'model\']);net.eval()\n            models[name].append((net,ck[\'scale\']))\n    return models,card\n\n\ndef predict_unlabelled_csv(csv_path,model_folder,device=\'cuda\'):\n    models,card=load_deployment(model_folder,device)\n    s=raw_to_stream(csv_path);s.update(file=Path(csv_path).name,pid=\'new\',part=\'inference\',events=[])\n    by_arch={name:average_predictions([predict(net,[s],scale,card[\'architectures\'][name],device=device) for net,scale in ens]) for name,ens in models.items()}\n    pred=combine_roles(by_arch,card[\'roles\'])[0];rows=[]\n    for c,ps,pf,valid in zip(pred[\'centers\'],pred[\'ps\'],pred[\'pf\'],pred[\'valid\']):\n        if not valid:state=label=\'unavailable\';confidence=0.\n        else:\n            state=STATES[int(ps.argmax())];confidence=float(ps.max())\n            label=\'idle\' if state==\'idle\' else (ACTIONS[pf[:12].argmax()] if state==\'activity\' else SIGNS[pf[12:].argmax()])\n        rows.append(dict(timestamp_us=int(s[\'t\'][c]),state=state,label=label,state_confidence=confidence))\n    return pd.DataFrame(rows),pd.DataFrame(commands(pred,card[\'tau\']))\n\n\nif __name__==\'__main__\':\n    import sys\n    if len(sys.argv)==3 and sys.argv[1]==\'--worker\':sys.exit(fast_worker(sys.argv[2]))\n', 'fyp_robust.py': '"""Fixed P1–P5 experiment. Import, preprocessing and inference never fit a model.\n\nReal fitting requires Kaggle + CUDA. The shared module preserves the previous\ndataset, coverage policy, event matching, class order and reporting definitions.\n"""\nfrom pathlib import Path\nimport copy\nimport gc\nimport json\nimport math\nimport os\nimport shutil\nimport sys\nimport time\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nimport fyp_common as C\n\nVERSION = \'p1p5-shape-spectrum-crossfit-v1\'\nARCH = dict(name=\'shape_spectrum4\', seconds=4, kind=\'shape_spectrum\')\nCFG = dict(batch_size=256, lr=8e-4, weight_decay=0.02, seed=17,\n           contrastive_weight=0.08, ema_decay=0.995, validate_seconds=25,\n           fit_seconds=540, checkpoint_seconds=50, restart_epochs=60,\n           final_seeds=[137,271])\nPIDS, ACTIONS, SIGNS, FINE = C.PIDS, C.ACTIONS, C.SIGNS, C.FINE\nPROGRESS = None\n\n\ndef inner_plan(out,work,streams,fingerprint,cfg,deadline):\n    out,work=Path(out),Path(work);jobs=[]\n    for ti,test in enumerate(PIDS):\n        for vi,val in enumerate(PIDS):\n            if val==test:continue\n            name=test+\'__val_\'+val\n            jobs.append(dict(pid=name,job=\'inner\',test=test,val=val,seed=17+ti*100+vi*13,\n                folder=str(out/test/\'models\'/(\'val_\'+val)),out=str(out),streams=str(streams),\n                fingerprint=fingerprint,cfg=dict(cfg),deadline=deadline,max_job_seconds=11*60,\n                spec_path=str(work/(name+\'.json\')),progress=str(work/(name+\'.progress.json\'))))\n    return jobs\n\n\ndef progress(phase):\n    if PROGRESS:\n        C.atomic_json(dict(time=time.time(), phase=phase), PROGRESS)\n\n\nclass Block(nn.Module):\n    def __init__(self, cin, cout, dilation=1):\n        super().__init__()\n        self.reduce=nn.Conv1d(cin,24,1,bias=False)\n        self.branches=nn.ModuleList([nn.Conv1d(24,24,k,padding=(k//2)*dilation,\n                                              dilation=dilation,bias=False) for k in [7,15,31]])\n        self.pool=nn.Sequential(nn.AvgPool1d(3,1,1),nn.Conv1d(cin,24,1,bias=False))\n        self.norm=nn.GroupNorm(8,96)\n        self.skip=nn.Conv1d(cin,cout,1,bias=False) if cin!=cout else nn.Identity()\n        self.se=nn.Sequential(nn.AdaptiveAvgPool1d(1),nn.Conv1d(cout,16,1),\n                              nn.GELU(),nn.Conv1d(16,cout,1),nn.Sigmoid())\n    def forward(self,x):\n        z=self.reduce(x)\n        h=self.norm(torch.cat([branch(z) for branch in self.branches]+[self.pool(x)],1))\n        return F.gelu(h*self.se(h)+self.skip(x))\n\n\nclass ShapeSpectrumCNN(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.temporal=nn.Sequential(Block(52,96),nn.AvgPool1d(2),Block(96,96),\n                                     nn.AvgPool1d(2),Block(96,96,dilation=2))\n        self.attention=nn.Sequential(nn.Conv1d(96,32,1),nn.Tanh(),nn.Conv1d(32,1,1))\n        self.spectral=nn.Sequential(nn.Linear(52*4,64),nn.LayerNorm(64),nn.GELU(),nn.Dropout(.2))\n        self.embedding=nn.Sequential(nn.Linear(96*3+64+2,192),nn.LayerNorm(192),nn.GELU(),nn.Dropout(.35))\n        self.detector=nn.Linear(192,3)\n        self.action=nn.Linear(192,len(ACTIONS));self.sign=nn.Linear(192,len(SIGNS))\n        self.project=nn.Sequential(nn.Linear(192,128),nn.GELU(),nn.Linear(128,64))\n        freq=torch.fft.rfftfreq(200,d=1/C.FS)\n        self.register_buffer(\'bands\',torch.stack([(freq>=lo)&(freq<hi) for lo,hi in [(0.5,2),(2,5),(5,10),(10,18.01)]]))\n\n    def forward(self,x,with_embedding=False):\n        # Spectral calculations stay in FP32 under mixed precision.\n        with torch.autocast(device_type=x.device.type,enabled=False):\n            xf=x.float();energy=xf.square().mean((1,2)).clamp_min(1e-5)\n            velocity=(xf[:,:,1:]-xf[:,:,:-1]).square().mean((1,2)).clamp_min(1e-5)\n            shape=(xf/energy.sqrt()[:,None,None]).clamp(-20,20)\n            spectrum=torch.fft.rfft(shape,dim=-1).abs().square()/shape.shape[-1]\n            bands=torch.stack([spectrum[:,:,mask].mean(-1) for mask in self.bands],-1)\n            bands=torch.log1p(bands).flatten(1)\n        h=self.temporal(shape)\n        w=self.attention(h).float().softmax(-1)\n        mean=(h*w).sum(-1)\n        std=((h.float()-mean[:,:,None]).square()*w).sum(-1).clamp_min(1e-5).sqrt()\n        z=self.embedding(torch.cat([mean,std,h.amax(-1),self.spectral(bands),\n                                    energy.log()[:,None],velocity.log()[:,None]],1))\n        st=self.detector(z)\n        # Each identity head has its own normalized distribution. The common\n        # reporting adapter\'s 26-way softmax gives each type half the mass.\n        fi=torch.cat([F.log_softmax(self.action(z).float(),1),F.log_softmax(self.sign(z).float(),1)],1)\n        return (st,fi,self.project(z)) if with_embedding else (st,fi)\n\n\ndef balanced_weights(streams,rows):\n    """Equal state, fine class, person, source/posture, then event contribution."""\n    keys=[]\n    for si,c,st,fi in rows:\n        s=streams[si];eid=int(s[\'event\'][c])\n        posture=s[\'events\'][eid][\'posture\'] if eid>=0 else \'transition\'\n        # Per-event balancing avoids a six-second repetition outweighing a\n        # four-second one. Unknown original postures remain unknown.\n        keys.append((int(st),int(fi),s[\'pid\'],s[\'part\'],posture,int(si),eid))\n    prefix_counts={}\n    for level in range(1,8):\n        prefix_counts[level]={}\n        for key in set(keys):\n            parent=key[:level-1]\n            prefix_counts[level].setdefault(parent,set()).add(key[:level])\n    counts=pd.Series(keys).value_counts().to_dict()\n    weights=[]\n    for key in keys:\n        probability=1.\n        for level in range(1,8): probability/=len(prefix_counts[level][key[:level-1]])\n        weights.append(probability/counts[key])\n    return torch.tensor(weights,dtype=torch.float32)\n\n\ndef make_bank(streams,device=\'cuda\'):\n    scale=C.fit_scale(streams)\n    rows=C.examples(streams)\n    offsets=np.cumsum([0]+[len(s[\'x\']) for s in streams[:-1]])\n    bank=torch.as_tensor(np.concatenate([np.clip(s[\'x\']/scale,-20,20) for s in streams]).T.copy(),device=device)\n    center=torch.as_tensor(rows[:,1]+offsets[rows[:,0]],device=device)\n    targets=torch.as_tensor(rows[:,2:],device=device)\n    pids=torch.as_tensor([PIDS.index(streams[si][\'pid\']) for si in rows[:,0]],device=device)\n    weights=balanced_weights(streams,rows).to(device)\n    return bank,center,targets,pids,weights,scale\n\n\ndef augment(x):\n    rms=x.square().mean((1,2),keepdim=True).sqrt().clamp_min(1e-3)\n    gain=torch.empty(len(x),1,1,device=x.device).uniform_(.75,1.35)\n    channel=torch.empty(len(x),52,1,device=x.device).uniform_(.9,1.1)\n    sign=torch.where(torch.rand(len(x),1,1,device=x.device)<.5,-1.,1.)\n    noise=torch.randn_like(x)*rms*torch.empty(len(x),1,1,device=x.device).uniform_(0,.025)\n    keep=(torch.rand(len(x),52,1,device=x.device)>.04)\n    return (x*gain*channel*sign+noise)*keep\n\n\ndef cross_person_contrastive(z,labels,people,temperature=.15):\n    """Same fine label in another TRAINING person supplies positive pairs."""\n    keep=labels>=0\n    z=F.normalize(z[keep].float(),dim=1);y=labels[keep];p=people[keep]\n    if len(z)<2:return z.sum()*0\n    logits=z@z.T/temperature\n    eye=torch.eye(len(z),device=z.device,dtype=torch.bool)\n    positive=(y[:,None]==y[None,:])&(p[:,None]!=p[None,:])&~eye\n    good=positive.any(1)\n    if not good.any():return z.sum()*0\n    logits=logits-logits.max(1,keepdim=True).values.detach()\n    logprob=logits-torch.logsumexp(logits.masked_fill(eye,-torch.inf),1,keepdim=True)\n    # masked_fill avoids 0 * inf on diagonal for very small synthetic batches.\n    sums=logprob.masked_fill(~positive,0).sum(1)\n    return -(sums[good]/positive.sum(1)[good]).mean()\n\n\ndef objective(st,fi,z,ys,yf,people,contrastive_weight=.08):\n    loss=.6*F.cross_entropy(st.float(),ys,label_smoothing=.02)\n    a=(ys==1)&(yf>=0);s=(ys==2)&(yf>=0)\n    if a.any():loss=loss+F.cross_entropy(fi[a,:12],yf[a],label_smoothing=.03)\n    if s.any():loss=loss+F.cross_entropy(fi[s,12:],yf[s]-12,label_smoothing=.03)\n    return loss+contrastive_weight*cross_person_contrastive(z,yf,people)\n\n\ndef validation_score(predictions):\n    table=C.segment_table(predictions);scores=[]\n    # Fixed before this run: story sign identity receives the largest weight.\n    for story,typ,names,weight in [(False,\'activity\',ACTIONS,.15),(False,\'sign\',SIGNS,.15),\n                                   (True,\'activity\',ACTIONS,.2),(True,\'sign\',SIGNS,.35)]:\n        g=table[table.type.eq(typ)&(table.part.eq(\'scripts\')==story)]\n        if len(g):scores.append((C.macro(g,names),weight))\n    g=table[table.part.eq(\'scripts\')]\n    if len(g):scores.append((C.f1_score(g.type,g.pred_state,labels=[\'activity\',\'sign\'],average=\'macro\',zero_division=0),.15))\n    return float(sum(s*w for s,w in scores)/sum(w for s,w in scores))\n\n\ndef update_ema(ema,net,decay):\n    with torch.no_grad():\n        for a,b in zip(ema.parameters(),net.parameters()):a.lerp_(b,1-decay)\n        for a,b in zip(ema.buffers(),net.buffers()):a.copy_(b)\n\n\ndef fit(tr,va,cfg,folder,fingerprint,deadline,epochs=None):\n    C.require_kaggle_gpu();torch.set_num_threads(2);C.seed_all(cfg[\'seed\'])\n    torch.backends.cudnn.benchmark=True\n    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)\n    progress(\'building GPU bank; training participants \'+\',\'.join(sorted({s[\'pid\'] for s in tr})))\n    bank,centers,targets,people,weights,scale=make_bank(tr)\n    net=ShapeSpectrumCNN().cuda();ema=copy.deepcopy(net).eval()\n    opt=torch.optim.AdamW(net.parameters(),lr=cfg[\'lr\'],weight_decay=cfg[\'weight_decay\'])\n    scheduler=torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt,T_0=cfg[\'restart_epochs\'],T_mult=2,eta_min=1e-5)\n    scaler=torch.amp.GradScaler(\'cuda\')\n    steps=max(32,math.ceil(len(centers)/cfg[\'batch_size\']))\n    positions=torch.arange(200,device=\'cuda\')-100\n    start=time.time();fit_end=min(deadline,start+cfg[\'fit_seconds\'])\n    best=-math.inf;best_epoch=0;best_steps=0;epoch=0;updates=0;history=[];full_epochs=0\n    last_validation=0.;last_checkpoint=0.;best_state=None\n    print(\'[FIT]\',folder,\'participants\',sorted({s[\'pid\'] for s in tr}),\n          \'validation\',sorted({s[\'pid\'] for s in va}),\'seconds\',round(fit_end-start),\n          \'batches_per_epoch\',steps,\'target_epochs\',epochs,flush=True)\n\n    def payload():\n        return dict(version=VERSION,arch=ARCH,classes=FINE,states=C.STATES,scale=scale,dataset=fingerprint,\n                    epoch=epoch,updates=updates,steps_per_epoch=steps,config=cfg,\n                    model={k:v.detach().cpu().clone() for k,v in ema.state_dict().items()},\n                    train_model={k:v.detach().cpu().clone() for k,v in net.state_dict().items()},\n                    optimizer=opt.state_dict(),scheduler=scheduler.state_dict(),scaler=scaler.state_dict(),\n                    rng_python=C.random.getstate(),rng_numpy=np.random.get_state(),rng_torch=torch.get_rng_state(),\n                    rng_cuda=torch.cuda.get_rng_state_all(),train_participants=sorted({s[\'pid\'] for s in tr}),\n                    validation_participants=sorted({s[\'pid\'] for s in va}),history=history)\n\n    while time.time()<fit_end and (epochs is None or epoch<epochs):\n        ep_start=time.time();net.train();loss_sum=0.;count=0\n        picks=torch.multinomial(weights,steps*cfg[\'batch_size\'],replacement=True).reshape(steps,cfg[\'batch_size\'])\n        for step,ix in enumerate(picks):\n            if step%8==0:\n                progress(f\'training epoch {epoch+1}, batch {step+1}/{steps}; updates {updates}\')\n                if time.time()>=fit_end:break\n            x=bank[:,centers[ix,None]+positions].permute(1,0,2).contiguous()\n            opt.zero_grad(set_to_none=True)\n            with torch.autocast(\'cuda\',dtype=torch.float16):\n                st,fi,z=net(augment(x),with_embedding=True)\n                loss=objective(st,fi,z,targets[ix,0],targets[ix,1],people[ix],cfg[\'contrastive_weight\'])\n            if not torch.isfinite(loss):raise RuntimeError(\'Nonfinite loss; last atomic checkpoint remains available.\')\n            scaler.scale(loss).backward();scaler.unscale_(opt)\n            torch.nn.utils.clip_grad_norm_(net.parameters(),1.)\n            old_scale=scaler.get_scale();scaler.step(opt);scaler.update()\n            if scaler.get_scale()>=old_scale:\n                update_ema(ema,net,cfg[\'ema_decay\']);updates+=1\n            scheduler.step(epoch+(step+1)/steps)\n            loss_sum+=float(loss.detach());count+=1\n        if not count:break\n        epoch+=1;full_epochs+=int(count==steps);score=None\n        need_val=bool(va) and (best_state is None or time.time()-last_validation>=cfg[\'validate_seconds\'] or time.time()>=fit_end)\n        if need_val:\n            progress(f\'validation at epoch {epoch}; test person remains excluded\')\n            score=validation_score(C.predict(ema,va,scale,ARCH,cfg[\'batch_size\']))\n            last_validation=time.time()\n        improved=not va or (score is not None and score>best+1e-5)\n        if improved:\n            best=score if score is not None else 0.;best_epoch=epoch;best_steps=updates\n            best_state={k:v.detach().cpu().clone() for k,v in ema.state_dict().items()}\n            C.checkpoint_save(dict(version=VERSION,arch=ARCH,classes=FINE,states=C.STATES,scale=scale,\n                dataset=fingerprint,epoch=epoch,updates=updates,model=best_state,validation_score=score,\n                train_participants=sorted({s[\'pid\'] for s in tr}),validation_participants=sorted({s[\'pid\'] for s in va})),folder/\'best.pt\')\n        history.append(dict(epoch=epoch,loss=loss_sum/count,validation_score=score,seconds=time.time()-ep_start,\n                            updates=updates,steps=count,full_epoch=count==steps,\n                            draws=count*cfg[\'batch_size\'],lr=opt.param_groups[0][\'lr\']))\n        if improved or time.time()-last_checkpoint>=cfg[\'checkpoint_seconds\'] or time.time()>=fit_end or epoch==epochs:\n            progress(f\'saving checkpoint epoch {epoch}\')\n            C.checkpoint_save(payload(),folder/\'last.pt\');last_checkpoint=time.time()\n            pd.DataFrame(history).to_csv(folder/\'history.csv\',index=False)\n        print(f\'[EPOCH] {folder.name} epoch={epoch} loss={loss_sum/count:.4f} val={score} updates={updates} elapsed={time.time()-start:.1f}s\',flush=True)\n    if best_state is None:raise RuntimeError(\'No model completed validation before the budget.\')\n    if updates==0:raise RuntimeError(\'No finite optimizer update completed; refusing to publish random weights.\')\n    C.checkpoint_save(payload(),folder/\'last.pt\');pd.DataFrame(history).to_csv(folder/\'history.csv\',index=False)\n    info=dict(best_epoch=best_epoch,best_updates=best_steps,epochs_completed=epoch,updates=updates,\n              steps_per_epoch=steps,training_seconds=time.time()-start,requested_epochs=epochs,\n              full_epochs_completed=full_epochs,target_completed=epochs is None or full_epochs>=epochs,validation_score=best,\n              train_participants=sorted({s[\'pid\'] for s in tr}),validation_participants=sorted({s[\'pid\'] for s in va}))\n    C.atomic_json(info,folder/\'fit_info.json\')\n    ema.load_state_dict(best_state)\n    del bank,centers,targets,people,weights,net,opt,scheduler,scaler\n    torch.cuda.empty_cache();gc.collect()\n    return ema,scale,info\n\n\ndef packed_predictions(predictions,path):\n    # Store frame predictions only. The common stream cache supplies true labels\n    # for evaluation, rather than copying raw recordings into every artifact.\n    C.save_predictions(predictions,path)\n\n\ndef restore_predictions(path,streams):\n    found={s[\'file\']:s for s in streams};result=[]\n    with np.load(path,allow_pickle=False) as z:\n        for i in range(len([k for k in z.files if k.endswith(\'_file\')])):\n            result.append(dict(stream=found[str(z[f\'{i}_file\'])],\n                               **{key:z[f\'{i}_{key}\'] for key in [\'centers\',\'ps\',\'pf\',\'valid\']}))\n    return result\n\n\ndef inner_worker(streams,spec):\n    test,val=spec[\'test\'],spec[\'val\']\n    assert test!=val and test in PIDS and val in PIDS\n    tr=[s for s in streams if s[\'pid\'] not in [test,val]]\n    va=[s for s in streams if s[\'pid\']==val];te=[s for s in streams if s[\'pid\']==test]\n    assert len({s[\'pid\'] for s in tr})==3\n    cfg=dict(spec[\'cfg\'],seed=spec[\'seed\']);folder=Path(spec[\'folder\'])\n    # Reserve time for inference and final atomic writes before supervisor cap.\n    net,scale,info=fit(tr,va,cfg,folder,spec[\'fingerprint\'],spec[\'deadline\']-75)\n    progress(\'saving validation predictions\')\n    packed_predictions(C.predict(net,va,scale,ARCH,cfg[\'batch_size\']),folder/\'validation_frames.npz\')\n    progress(\'saving untouched outer test predictions\')\n    packed_predictions(C.predict(net,te,scale,ARCH,cfg[\'batch_size\']),folder/\'test_frames.npz\')\n    C.atomic_json(dict(test=test,validation=val,seed=cfg[\'seed\'],arch=ARCH,dataset=spec[\'fingerprint\'],\n                       training_participants=sorted({s[\'pid\'] for s in tr}),**info),folder/\'complete.json\')\n    C.release(net);print(\'[MEMBER COMPLETE]\',test,val,flush=True)\n\n\n_ORIGINAL_COMMANDS=C.commands\ndef commands(p,tau):\n    # Keep original smoothing, boundary handling and matching; reject ambiguous\n    # predicted sign intervals without changing the coverage denominator.\n    minimum=p.get(\'sign_confidence\',0.)\n    all_commands=_ORIGINAL_COMMANDS(p,tau)\n    if minimum<=0:return all_commands\n    selected=[]\n    for command in all_commands:\n        m=(p[\'centers\']>=command[\'i0\'])&(p[\'centers\']<command[\'i1\'])&p[\'valid\']\n        if not m.any():continue\n        probabilities=p[\'pf\'][m,12:].mean(0)\n        probabilities=probabilities/max(float(probabilities.sum()),1e-12)\n        if probabilities.max()>=minimum:selected.append(command)\n    return selected\nC.commands=commands\n\n\ndef choose_operating_point(validation_sets):\n    rows=[]\n    for tau in np.arange(.3,.901,.1):\n        for confidence in [0.,.2,.35,.5]:\n            individual=[]\n            for predictions in validation_sets:\n                for p in predictions:p[\'sign_confidence\']=confidence\n                r,_=C.event_metrics(predictions,float(tau))\n                precision=r[\'identity_precision\'];recall=r[\'identity_recall\']\n                r[\'identity_f0_5\']=1.25*precision*recall/max(.25*precision+recall,1e-12)\n                individual.append(r)\n            rows.append(dict(tau=float(tau),confidence=confidence,\n                **{key:float(np.mean([r[key] for r in individual])) for key in\n                   [\'identity_f0_5\',\'identity_f1\',\'detection_f1\',\'false_commands_per_routine_hour\']}))\n    # Precision receives extra weight. Selection never examines test labels.\n    best=max(rows,key=lambda r:(r[\'identity_f0_5\'],r[\'identity_f1\'],\n                               -r[\'false_commands_per_routine_hour\'],r[\'detection_f1\'],r[\'tau\'],r[\'confidence\']))\n    return best,rows\n\n\ndef score_outer(spec,streams):\n    out=Path(spec[\'out\']);test=spec[\'test\'];fold=out/test\n    members=[fold/\'models\'/(\'val_\'+val) for val in PIDS if val!=test]\n    assert all((m/\'complete.json\').exists() for m in members),f\'{test}: missing inner member; no partial ensemble score is published.\'\n    for member in members:\n        meta=json.loads((member/\'complete.json\').read_text());val=member.name.removeprefix(\'val_\')\n        assert meta[\'test\']==test and meta[\'validation\']==val\n        expected=set(PIDS)-{test,val}\n        assert set(meta[\'train_participants\'])==expected and set(meta[\'validation_participants\'])=={val}\n        if \'fingerprint\' in spec:assert meta[\'dataset\']==spec[\'fingerprint\']\n    validation_sets=[restore_predictions(m/\'validation_frames.npz\',streams) for m in members]\n    operating,grid=choose_operating_point(validation_sets)\n    pd.DataFrame(grid).to_csv(fold/\'validation_operating_points.csv\',index=False)\n    predictions=C.average_predictions([restore_predictions(m/\'test_frames.npz\',streams) for m in members])\n    for p in predictions:p[\'sign_confidence\']=operating[\'confidence\']\n    C.atomic_json(dict(test=test,operating_point=operating,members=[str(m.relative_to(out)) for m in members],\n        selection=\'Four cross-fitted validation people; correct-identity F0.5 objective; no outer test labels.\'),fold/\'selection.json\')\n    C.save_predictions(predictions,fold/\'frame_predictions.npz\')\n    C.report_fold(C.segment_table(predictions),predictions,operating[\'tau\'],fold)\n    C.atomic_json(dict(test=test,version=VERSION,members=4,training=\'Each member trains on 3 of the 4 other people.\'),fold/\'complete.json\')\n    print(\'[OUTER TEST COMPLETE]\',test,flush=True)\n\n\ndef final_worker(streams,spec):\n    cfg=dict(spec[\'cfg\'],seed=spec[\'seed\'],fit_seconds=spec[\'final_seconds\'])\n    net,scale,info=fit(streams,[],cfg,spec[\'folder\'],spec[\'fingerprint\'],spec[\'deadline\']-25,epochs=spec[\'epochs\'])\n    C.atomic_json(info,Path(spec[\'folder\'])/\'complete.json\');C.release(net)\n    print(\'[FINAL SEED COMPLETE]\',spec[\'seed\'],info,flush=True)\n\n\ndef package_deployment(out,fingerprint,cfg):\n    out=Path(out);final=out/\'deployment_all5\'\n    checkpoints=[final/\'models\'/f\'seed_{seed}\'/\'best.pt\' for seed in cfg[\'final_seeds\']]\n    assert all(p.exists() and (p.parent/\'complete.json\').exists() for p in checkpoints),\'All-five models are incomplete.\'\n    selections=[json.loads((out/pid/\'selection.json\').read_text()) for pid in PIDS]\n    infos=[json.loads((p.parent/\'fit_info.json\').read_text()) for p in checkpoints]\n    card=dict(version=VERSION,architecture=ARCH,classes=FINE,states=C.STATES,fs=C.FS,dataset=fingerprint,\n        models=[str(p.relative_to(final)) for p in checkpoints],training_participants=PIDS,\n        tau=float(np.median([s[\'operating_point\'][\'tau\'] for s in selections])),\n        sign_confidence=float(np.median([s[\'operating_point\'][\'confidence\'] for s in selections])),\n        final_fit_info=infos,selected_epoch_targets_completed=all(x[\'target_completed\'] for x in infos),\n        evaluation=\'Five outer test ensembles of four cross-fit models. Final two-seed all-five refit has no independent score.\',\n        label_protocol=\'Cue/key-press labels; video alignment unverified. No test-person calibration.\',\n        inference=\'Offline centered four-second windows; 50 Hz; CSI magnitude only; conditional action/sign heads.\',\n        threshold_note=\'Final median cross-fit operating points; calibration drift in final ensemble is unverified.\')\n    C.atomic_json(card,final/\'model_card.json\')\n    for name in [\'fyp_robust.py\',\'fyp_common.py\']:shutil.copy2(Path(__file__).parent/name,final/name)\n    (final/\'README.md\').write_text(\'\'\'# All-five CSI model\n\nInstall torch, numpy, pandas, scipy, scikit-learn and joblib. Keep this folder structure intact.\nInference: add this directory to sys.path, import fyp_robust, then call\n`frames, commands = fyp_robust.predict_unlabelled_csv("new_csi.csv", model_folder, device="cuda")`.\nUse `device="cpu"` for slower inference. The input must use the original 107-column CSI schema.\nThe final all-five model has no independent test score; see model_card.json and the outer held-out results.\n\'\'\')\n    import zipfile\n    path=out.parent/\'FYP_Improved_All5_Model.zip\';tmp=path.with_suffix(\'.zip.part\')\n    with zipfile.ZipFile(tmp,\'w\',zipfile.ZIP_DEFLATED,compresslevel=1) as z:\n        for p in final.rglob(\'*\'):\n            if p.is_file() and p.name not in [\'last.pt\',\'last.previous.pt\',\'best.previous.pt\',\'history.csv\']:\n                z.write(p,Path(\'FYP_Improved_All5_Model\')/p.relative_to(final))\n    os.replace(tmp,path);C.atomic_json(dict(sha256=C.sha256(path),bytes=path.stat().st_size),path.with_suffix(\'.zip.json\'))\n    return card\n\n\ndef load_deployment(folder,device=\'cuda\'):\n    folder=Path(folder);card=json.loads((folder/\'model_card.json\').read_text());models=[]\n    assert card[\'classes\']==FINE\n    for relative in card[\'models\']:\n        ck=C.trusted_load(folder/relative)\n        assert ck[\'classes\']==FINE\n        net=ShapeSpectrumCNN().to(device);net.load_state_dict(ck[\'model\']);net.eval()\n        models.append((net,ck[\'scale\']))\n    return models,card\n\n\ndef predict_unlabelled_csv(csv_path,model_folder,device=\'cuda\'):\n    models,card=load_deployment(model_folder,device)\n    stream=C.raw_to_stream(csv_path);stream.update(file=Path(csv_path).name,pid=\'new\',part=\'inference\',events=[])\n    p=C.average_predictions([C.predict(net,[stream],scale,ARCH,device=device) for net,scale in models])[0]\n    p[\'sign_confidence\']=card[\'sign_confidence\'];rows=[]\n    for center,st,fi,valid in zip(p[\'centers\'],p[\'ps\'],p[\'pf\'],p[\'valid\']):\n        state=C.STATES[int(st.argmax())] if valid else \'unavailable\'\n        label=\'idle\' if state==\'idle\' else (ACTIONS[fi[:12].argmax()] if state==\'activity\' else\n                                          SIGNS[fi[12:].argmax()] if state==\'sign\' else \'unavailable\')\n        rows.append(dict(timestamp_us=int(stream[\'t\'][center]),state=state,label=label))\n    return pd.DataFrame(rows),pd.DataFrame(commands(p,card[\'tau\']))\n\n\ndef worker(path):\n    global PROGRESS\n    spec=json.loads(Path(path).read_text());PROGRESS=spec[\'progress\']\n    C.require_kaggle_gpu();progress(\'loading preprocessed streams\')\n    streams=joblib.load(spec[\'streams\'])\n    if spec[\'job\']==\'inner\':inner_worker(streams,spec)\n    elif spec[\'job\']==\'final\':final_worker(streams,spec)\n    else:raise ValueError(spec[\'job\'])\n\n\nif __name__==\'__main__\':worker(sys.argv[2])\n', 'run_jobs.py': '"""Process supervision for bounded Kaggle runs. This module never trains on import."""\nfrom pathlib import Path\nimport os,sys,json,time,signal,subprocess,collections\n\n\ndef stop(proc):\n    if proc.poll() is not None:return\n    os.killpg(proc.pid,signal.SIGTERM)\n    try:proc.wait(timeout=5)\n    except subprocess.TimeoutExpired:\n        os.killpg(proc.pid,signal.SIGKILL);proc.wait(timeout=5)\n\n\ndef launch(spec,script):\n    path=Path(spec[\'spec_path\']);path.parent.mkdir(parents=True,exist_ok=True)\n    path.write_text(json.dumps(spec,indent=2));log=path.with_suffix(\'.log\')\n    env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(spec[\'gpu\']),PYTHONUNBUFFERED=\'1\',\n             OMP_NUM_THREADS=\'2\',OPENBLAS_NUM_THREADS=\'2\',MKL_NUM_THREADS=\'2\')\n    file=open(log,\'w\')\n    proc=subprocess.Popen([sys.executable,\'-u\',str(script),\'--worker\',str(path)],stdout=file,\n        stderr=subprocess.STDOUT,env=env,start_new_session=True)\n    return dict(spec=spec,proc=proc,file=file,log=log,pos=0,started=time.time(),notice=time.time())\n\n\ndef drain(job):\n    with open(job[\'log\']) as f:\n        f.seek(job[\'pos\']);s=f.read();job[\'pos\']=f.tell()\n    if s:\n        for line in s.splitlines():print(f\'[{job["spec"]["pid"]}] {line}\',flush=True)\n\n\ndef run_jobs(specs,script,max_parallel=2,stall_seconds=300,poll_seconds=2):\n    pending=collections.deque(specs);active=[];results=[];free=list(range(max_parallel))\n    try:\n        while pending or active:\n            while pending and free:\n                spec=dict(pending.popleft())\n                if time.time()>=spec[\'deadline\']:\n                    results.append(dict(pid=spec[\'pid\'],status=\'not_started_time_budget\'));continue\n                gpu=free.pop(0);spec[\'gpu\']=gpu\n                if spec.get(\'max_job_seconds\'):\n                    spec[\'deadline\']=min(spec[\'deadline\'],time.time()+spec[\'max_job_seconds\'])\n                print(\'[START]\',spec[\'pid\'],\'GPU\',gpu,flush=True)\n                active.append(launch(spec,script))\n            for job in list(active):\n                drain(job);spec=job[\'spec\'];now=time.time();rc=job[\'proc\'].poll();reason=None\n                last=job[\'started\'];phase=\'worker startup\'\n                progress=Path(spec[\'progress\'])\n                if progress.exists():\n                    try:\n                        data=json.loads(progress.read_text());last=data[\'time\'];phase=data[\'phase\']\n                    except (OSError,ValueError,KeyError):pass\n                if rc is None and now>=spec[\'deadline\']:\n                    reason=\'time_budget_reached\';stop(job[\'proc\']);rc=job[\'proc\'].returncode\n                elif rc is None and now-last>=stall_seconds:\n                    reason=\'stalled_no_progress\';stop(job[\'proc\']);rc=job[\'proc\'].returncode\n                if rc is not None:\n                    drain(job);job[\'file\'].close();active.remove(job);free.append(spec[\'gpu\']);free.sort()\n                    result=dict(pid=spec[\'pid\'],status=reason or (\'complete\' if rc==0 else \'failed_or_paused\'),\n                        exit_code=rc,seconds=round(now-job[\'started\'],1),last_phase=phase,log=str(job[\'log\']))\n                    results.append(result)\n                    Path(spec[\'spec_path\']).with_suffix(\'.status.json\').write_text(json.dumps(result,indent=2))\n                    print(\'[END]\',result,flush=True)\n                elif now-job[\'notice\']>=60:\n                    print(\'[STATUS]\',spec[\'pid\'],phase,\'seconds since progress:\',int(now-last),flush=True)\n                    job[\'notice\']=now\n            if pending or active:time.sleep(poll_seconds)\n    finally:\n        for job in active:\n            stop(job[\'proc\']);drain(job);job[\'file\'].close()\n    return results\n', 'prepare_data.py': "from pathlib import Path\nimport sys,json,time,joblib,hashlib\nimport fyp_common as fyp\n\nKNOWN_ROOT=Path('/kaggle/input/datasets/adhikkeshsk/fyp-dataset/phase1')\n\ndef resolve_input(cfg,input_base=Path('/kaggle/input')):\n    input_base=Path(input_base)\n    if cfg.get('root'):\n        root=Path(cfg['root'])\n        if not (root/'session_log.csv').is_file() and (root/'phase1/session_log.csv').is_file():root=root/'phase1'\n        if not (root/'manifest.json').is_file() or not (root/'session_log.csv').is_file():\n            raise FileNotFoundError(f'DATA_ROOT={root} is not a mounted verified CSI dataset. Attach fyp-dataset using Add Input, then copy its phase1 folder path.')\n        return root,None\n    if cfg.get('zip'):\n        archive=Path(cfg['zip'])\n        if not archive.is_file():raise FileNotFoundError(f'ZIP_PATH does not exist: {archive}. Attach the verified data ZIP as Input first.')\n        return None,archive\n    candidates=sorted({p.parent.resolve() for p in input_base.rglob('manifest.json') if (p.parent/'session_log.csv').is_file()})\n    if len(candidates)==1:return candidates[0],None\n    if KNOWN_ROOT in candidates:return KNOWN_ROOT,None\n    if len(candidates)>1:\n        hashes={hashlib.sha256((p/'manifest.json').read_bytes()).hexdigest() for p in candidates}\n        if len(hashes)==1:return candidates[0],None\n        raise ValueError('Multiple different CSI datasets are attached. Set DATA_ROOT to one phase1 folder: '+', '.join(map(str,candidates)))\n    archives=list(input_base.rglob('FYP_Phase1_P01_P05_verified.zip'))\n    if len(archives)==1:return None,archives[0]\n    if len(archives)>1:raise ValueError('Multiple data ZIPs are attached. Set ZIP_PATH: '+', '.join(map(str,archives)))\n    mounts=sorted(str(p) for p in input_base.iterdir()) if input_base.exists() else []\n    raise FileNotFoundError('No P1-P5 CSI dataset was found in this notebook. Add Input -> Your Datasets -> fyp-dataset -> Add. '\n        'The previous successful path was '+str(KNOWN_ROOT)+'. Current input mounts: '+str(mounts))\n\n\ndef main(spec_path):\n    spec=json.loads(Path(spec_path).read_text());cfg=spec['data'];work=Path(spec['work']);work.mkdir(parents=True,exist_ok=True)\n    fyp.atomic_json(dict(time=time.time(),phase='finding attached dataset'),spec['progress'])\n    root,archive=resolve_input(cfg)\n    if archive is not None:root=fyp.extract_verified(archive,Path('/kaggle/temp/fyp_robust_data'),cfg['sha'])\n    print('[DATASET]',root,flush=True)\n    fyp.atomic_json(dict(time=time.time(),phase='validating and preprocessing real recordings'),spec['progress'])\n    streams,fingerprint=fyp.prepare(root,work)\n    cache=work/'cache';cache.mkdir(exist_ok=True)\n    joblib.dump(streams,cache/'prepared_streams.joblib',compress=0)\n    fyp.atomic_json(dict(fingerprint=fingerprint,streams=len(streams),participants=sorted({s['pid'] for s in streams}),dataset_root=str(root)),work/'prepared.json')\n    fyp.atomic_json(dict(time=time.time(),phase='preprocessing complete'),spec['progress'])\n    print('[DATA READY]',fingerprint,len(streams),'recordings',flush=True)\n\n\nif __name__=='__main__':main(sys.argv[2])\n", 'assemble_evaluation.py': '"""Combine saved members and choose thresholds using inner validation only."""\nfrom pathlib import Path\nimport json\nimport sys\nimport time\nimport joblib\nimport fyp_robust as R\n\ndef main(spec):\n    streams=joblib.load(spec[\'streams\']);done=[];missing=[]\n    for test in R.PIDS:\n        R.C.atomic_json(dict(time=time.time(),phase=\'assembling held-out \'+test),spec[\'progress\'])\n        try:R.score_outer(dict(spec,test=test),streams);done.append(test)\n        except AssertionError as exc:missing.append(str(exc));print(\'[INCOMPLETE]\',exc,flush=True)\n    R.C.atomic_json(dict(completed=done,missing=missing),Path(spec[\'out\'])/\'evaluation_assembly.json\')\n\nif __name__==\'__main__\':main(json.loads(Path(sys.argv[2]).read_text()))\n', 'finish_results.py': '"""Reporting, common evaluation, packaging. No fitting."""\nfrom pathlib import Path\nimport hashlib\nimport json\nimport os\nimport sys\nimport time\n\nos.environ[\'MPLBACKEND\']=\'Agg\'\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport joblib\nimport fyp_robust as R\n\n\ndef report(spec):\n    C=R.C;out=Path(spec[\'out\']);streams=joblib.load(spec[\'streams\'])\n    progress=spec[\'progress\']\n    def beat(phase):C.atomic_json(dict(time=time.time(),phase=phase),progress)\n    completed=[];errors=[]\n    for test in C.PIDS:\n        beat(\'assembling untouched outer test \'+test)\n        try:\n            if not (out/test/\'complete.json\').exists():R.score_outer(dict(spec,test=test),streams)\n            completed.append(test)\n        except AssertionError as exc:errors.append(str(exc));print(\'[INCOMPLETE]\',exc,flush=True)\n    if completed:\n        beat(\'class-wise results and confusion matrices\')\n        C.summarize(out)\n        reference=json.loads((Path(__file__).parent/\'baseline_reference.json\').read_text())\n        data=pd.read_csv(out/\'all_out_of_fold_predictions.csv\')\n        ids_hash=hashlib.sha256(\'\\n\'.join(sorted(data.id)).encode()).hexdigest()\n        info=json.loads((out/\'dataset_info.json\').read_text())\n        matched=len(completed)==5 and ids_hash==reference[\'event_ids_sha256\'] and info[\'dataset\']==reference[\'dataset\']\n        if matched:\n            old=pd.DataFrame(reference[\'summary\'])\n            new=pd.read_csv(out/\'summary.csv\')\n            comparison=old.merge(new,on=[\'split\',\'type\'],suffixes=(\'_baseline\',\'_new\'),validate=\'one_to_one\')\n            comparison[\'change_pp\']=100*(comparison.mean_f1_new-comparison.mean_f1_baseline)\n            comparison.to_csv(out/\'comparison_with_balanced.csv\',index=False)\n            print(\'[SAME-EVENT COMPARISON]\\n\',comparison[[\'split\',\'type\',\'mean_f1_baseline\',\'mean_f1_new\',\'change_pp\']].to_string(index=False),flush=True)\n            fig,ax=plt.subplots(figsize=(11,5))\n            g=comparison[comparison.split.isin([\'isolated\',\'stories_oracle_boundaries\'])]\n            x=np.arange(len(g));ax.bar(x-.19,g.mean_f1_baseline,.38,label=\'Previous balanced\')\n            ax.bar(x+.19,g.mean_f1_new,.38,label=\'New cross-fit ensemble\')\n            ax.set_xticks(x,[f\'{a.replace("stories_oracle_boundaries","Story, known bounds")}\\n{b}\' for a,b in zip(g.split,g.type)])\n            ax.set_ylim(0,1);ax.set_ylabel(\'Mean held-out macro F1\');ax.set_title(\'Same dataset and test event IDs; training protocols differ\')\n            ax.legend();fig.tight_layout();fig.savefig(out/\'figures/previous_vs_new.png\',dpi=170);plt.close(fig)\n        else:\n            errors.append(\'Automatic baseline comparison withheld: dataset/event IDs or evaluation coverage differ.\')\n        # Learning curves show validation, rather than only training loss.\n        fig,axes=plt.subplots(1,2,figsize=(13,5))\n        for pid in completed:\n            for p in (out/pid/\'models\').glob(\'val_*/history.csv\'):\n                d=pd.read_csv(p);label=pid+\'/\'+p.parent.name\n                axes[0].plot(d.epoch,d.loss,alpha=.6,label=label)\n                valid=d.validation_score.notna();axes[1].plot(d.loc[valid,\'epoch\'],d.loc[valid,\'validation_score\'],alpha=.7,label=label)\n        axes[0].set_title(\'Optimization loss\');axes[1].set_title(\'Validation selection score — no test data\')\n        for ax in axes:ax.set_xlabel(\'Epoch\')\n        axes[1].legend(ncol=2,fontsize=6);fig.tight_layout();fig.savefig(out/\'figures/crossfit_learning_curves.png\',dpi=170);plt.close(fig)\n    status=dict(completed_test_participants=completed,evaluation_complete=len(completed)==5,\n                all_five_model_ready=(out/\'deployment_all5/model_card.json\').exists(),errors=errors,\n                elapsed_minutes=(time.time()-spec[\'started\'])/60,\n                evaluation=\'Updated benchmark informed by previous tests; not a fresh independent study.\')\n    if status[\'all_five_model_ready\']:\n        card=json.loads((out/\'deployment_all5/model_card.json\').read_text())\n        status[\'final_selected_epoch_targets_completed\']=card[\'selected_epoch_targets_completed\']\n    C.atomic_json(status,out/\'run_completion.json\')\n    beat(\'exporting checkpoint archive\')\n    archive=C.archive_outputs(out)\n    beat(\'report and archive complete\')\n    print(\'[FINISHED]\',json.dumps(status,indent=2),flush=True)\n    print(\'[ARCHIVE]\',archive,flush=True)\n\n\nif __name__==\'__main__\':report(json.loads(Path(sys.argv[2]).read_text()))\n', 'baseline_reference.json': '{\n  "name": "FYP_BALANCED_2H_20261017_015419",\n  "dataset": "f01651df3c4b758f3cbec5dd742669b4f0605a9f11c24162aecd662949dede62",\n  "events": 1534,\n  "event_ids_sha256": "6d6407f44d9057ce65f518f00a9aed7f76bcad761b3d8f39e182a541c4585211",\n  "summary": [\n    {\n      "split": "isolated",\n      "type": "activity",\n      "mean_f1": 0.502294148782432,\n      "sd_f1": 0.0841145544785871,\n      "participants": 5,\n      "events": 628\n    },\n    {\n      "split": "isolated",\n      "type": "sign",\n      "mean_f1": 0.3323200776397995,\n      "sd_f1": 0.0296305464643273,\n      "participants": 5,\n      "events": 490\n    },\n    {\n      "split": "isolated_original_parts",\n      "type": "activity",\n      "mean_f1": 0.5304250454701405,\n      "sd_f1": 0.0726291286189167,\n      "participants": 5,\n      "events": 488\n    },\n    {\n      "split": "isolated_original_parts",\n      "type": "sign",\n      "mean_f1": 0.3654160438068,\n      "sd_f1": 0.0556403666728021,\n      "participants": 5,\n      "events": 350\n    },\n    {\n      "split": "isolated_seated",\n      "type": "activity",\n      "mean_f1": 0.1939172697050848,\n      "sd_f1": 0.0876035655165751,\n      "participants": 2,\n      "events": 140\n    },\n    {\n      "split": "isolated_seated",\n      "type": "sign",\n      "mean_f1": 0.1995313675709518,\n      "sd_f1": 0.065264149453516,\n      "participants": 2,\n      "events": 140\n    },\n    {\n      "split": "stories_oracle_boundaries",\n      "type": "activity",\n      "mean_f1": 0.523659281650661,\n      "sd_f1": 0.1354685699724965,\n      "participants": 5,\n      "events": 208\n    },\n    {\n      "split": "stories_oracle_boundaries",\n      "type": "sign",\n      "mean_f1": 0.247111975326261,\n      "sd_f1": 0.0761250961173636,\n      "participants": 5,\n      "events": 208\n    }\n  ]\n}'}
for filename,source in SOURCES.items():(SOURCE_DIR/filename).write_text(source)
sys.path.insert(0,str(SOURCE_DIR))
import fyp_robust as R
import run_jobs
print('Experiment:',R.VERSION,'fixed model:',R.ARCH,flush=True)


## Verify and prepare the real P1–P5 recordings

The manifest checks every dataset file. This uses the same input resolver as the successful balanced run. Partial P2 labels are excluded; unavailable covered events count as misses. Only P4/P5 contain the new seated/cooking recordings. Empty-room recordings are quality references. No normalization is fitted on the test person.


In [ ]:
prep=dict(pid='PREPARE',spec_path=str(WORK/'prepare.json'),progress=str(WORK/'prepare.progress.json'),
          deadline=min(RUN_STARTED+7*60,EVALUATION_DEADLINE),work=str(WORK),
          data=dict(root=DATA_ROOT,zip=ZIP_PATH,sha=EXPECTED_ZIP_SHA256))
prepared_status=run_jobs.run_jobs([prep],SOURCE_DIR/'prepare_data.py',max_parallel=1,stall_seconds=300)
assert prepared_status[0]['status']=='complete', 'Preparation failed. Read prepare.log; no training started.'
prepared=json.loads((WORK/'prepared.json').read_text())
assert prepared['participants']==R.PIDS and prepared['streams']==22
for name in ['recording_audit.csv','event_coverage.csv','dataset_info.json']:
    shutil.copy2(WORK/name,OUT/name)
for name in SOURCES:shutil.copy2(SOURCE_DIR/name,OUT/name)
versions={m:importlib.import_module(m).__version__ for m in ['torch','numpy','pandas','scipy','sklearn']}
R.C.atomic_json(versions,OUT/'environment.json')
display(pd.read_csv(OUT/'recording_audit.csv'))
coverage=pd.read_csv(OUT/'event_coverage.csv')
display(coverage.groupby(['pid','part','covered']).size().rename('events').reset_index())
cfg=dict(R.CFG)
R.C.atomic_json(cfg,OUT/'training_config.json')
common=dict(out=str(OUT),streams=str(WORK/'cache/prepared_streams.joblib'),fingerprint=prepared['fingerprint'],
            cfg=cfg,started=RUN_STARTED)


## Train twenty cross-fit members — approximately 90 minutes

One GPU runs one independent job. Each job has nine minutes for real optimization, validation and atomic checkpoint updates. The loop continues for its assigned budget, with a fixed cosine-restart learning-rate schedule; it keeps the highest validation-scoring moving-average checkpoint, even if later epochs overfit. Training time itself is not evidence of quality.

Every outer test person is excluded from four member fits. Each member uses a different validation person. The validation score gives extra weight to story sign identity, using a fixed formula recorded in the code. Sampling is with replacement; a dataset-sized draw count does not mean every unique window was visited.

Logs print epochs, loss, validation score and update counts. Status appears at least every minute. `best.pt`, recoverable `last.pt`, histories and participant metadata are saved during fitting. No DataLoader workers or DataParallel are used.


In [ ]:
jobs=R.inner_plan(OUT,WORK,common['streams'],common['fingerprint'],cfg,EVALUATION_DEADLINE)
assert len(jobs)==20
display(pd.DataFrame([dict(test=j['test'],validation=j['val'],seed=j['seed'],
    train=','.join(p for p in R.PIDS if p not in [j['test'],j['val']])) for j in jobs]))
member_status=run_jobs.run_jobs(jobs,SOURCE_DIR/'fyp_robust.py',max_parallel=2,stall_seconds=300)
R.C.atomic_json(member_status,OUT/'member_status.json')
display(pd.DataFrame(member_status))


## Assemble five untouched outer test scores

Each threshold/confidence pair is chosen from four validation people whose predictions came from models that excluded them. The operating-point objective favors correctly identified signs and precision (F0.5). The four test predictions are averaged, then scored once. Confidence rejection leaves the measured routine-time denominator unchanged.

These ensemble members each fit three people, unlike the previous run’s four-person refit. The comparison measures the complete methods, not architecture alone. Incomplete four-member ensembles are not reported as completed folds.


In [ ]:
assembly_spec=dict(common,pid='ASSEMBLE',spec_path=str(WORK/'assemble.json'),progress=str(WORK/'assemble.progress.json'),
                   deadline=min(TRAINING_DEADLINE,time.time()+5*60))
assembly_status=run_jobs.run_jobs([assembly_spec],SOURCE_DIR/'assemble_evaluation.py',max_parallel=1,stall_seconds=300)
R.C.atomic_json(assembly_status,OUT/'assembly_status.json')
completed=[pid for pid in R.PIDS if (OUT/pid/'complete.json').exists()]
print('Completed untouched participant tests:',completed,flush=True)


## Train and save the all-five deployment models

Two new seeds train on all five people. Their epoch target is the median best epoch from the twenty inner validation fits; they use the same learning-rate schedule and dataset-scaled batches. The remaining time caps these refits. A budget-shortened target is recorded explicitly in `model_card.json`, rather than hidden.

The deployment package contains both selected weights, class order, normalization scales, model code and an inference example. This all-five refit has no separate independent test score; the reported results above belong to the four-member outer ensembles. Centered windows make this an offline prototype.


In [ ]:
final_status=[]
if len(completed)==5:
    import numpy as np
    infos=[json.loads((Path(j['folder'])/'fit_info.json').read_text()) for j in jobs]
    final_epochs=max(1,int(np.median([info['best_epoch'] for info in infos])))
    remaining=TRAINING_DEADLINE-time.time()-90
    if remaining>60:
        final_seconds=min(8*60,remaining)
        final_jobs=[]
        for seed in cfg['final_seeds']:
            name='FINAL_'+str(seed)
            final_jobs.append(dict(common,pid=name,job='final',seed=seed,epochs=final_epochs,final_seconds=final_seconds,
                folder=str(OUT/'deployment_all5/models'/('seed_'+str(seed))),
                deadline=TRAINING_DEADLINE,spec_path=str(WORK/(name+'.json')),progress=str(WORK/(name+'.progress.json'))))
        print('All-five target:',final_epochs,'epochs; cap per seed:',round(final_seconds/60,1),'minutes',flush=True)
        final_status=run_jobs.run_jobs(final_jobs,SOURCE_DIR/'fyp_robust.py',max_parallel=2,stall_seconds=300)
        if all(s['status']=='complete' for s in final_status):
            card=R.package_deployment(OUT,prepared['fingerprint'],cfg)
            display(card)
        else:print('A final seed failed; checkpoints remain, but no ready deployment package is declared.',flush=True)
    else:print('Insufficient final-training budget. Evaluation and checkpoint files are preserved.',flush=True)
else:print('Final refit withheld because all five four-member tests did not complete.',flush=True)
R.C.atomic_json(final_status,OUT/'final_status.json')


## Class graphs, baseline comparison and download

Outputs include action/sign class F1, confusion matrices, participant and posture results, continuous detection, correct identity, full intent and false commands. The automatic comparison to the previous balanced run is allowed only when the dataset fingerprint and all event IDs match. Negative changes are kept.

Download `FYP_Improved_All5_Model.zip` for the inference model, and `FYP_all_checkpoints_and_results.zip` for models, histories and results. Output files are also saved individually. Keep the original dataset and the completed Kaggle notebook version.


In [ ]:
report_spec=dict(common,pid='REPORT',spec_path=str(WORK/'report.json'),progress=str(WORK/'report.progress.json'),
                 deadline=FINISH_DEADLINE)
report_status=run_jobs.run_jobs([report_spec],SOURCE_DIR/'finish_results.py',max_parallel=1,stall_seconds=300)
R.C.atomic_json(report_status,OUT/'report_status.json')
if (OUT/'run_completion.json').exists():
    status=json.loads((OUT/'run_completion.json').read_text());display(status)
else:status=dict(evaluation_complete=False,all_five_model_ready=False,error='Report did not complete; inspect report.log.')
for name in ['FYP_all_checkpoints_and_results.zip','FYP_Improved_All5_Model.zip']:
    path=WORK/name
    if path.exists():display(FileLink(str(path)))
for name in ['summary.csv','comparison_with_balanced.csv','story_metrics.csv']:
    if (OUT/name).exists():print(name);display(pd.read_csv(OUT/name))
for figure in sorted((OUT/'figures').glob('*.png')):display(Image(filename=str(figure)))
print('Total elapsed:',round((time.time()-RUN_STARTED)/60,1),'minutes. Saved under:',WORK,flush=True)
assert status['evaluation_complete'] and status['all_five_model_ready'], 'INCOMPLETE RUN: saved outputs remain available; read member/assembly/final/report status logs.'


## Validation limits

Before delivery, local checks cover notebook syntax/schema, embedded source consistency, model forward passes, finite conditional losses, cross-person contrastive masking, source/event balancing, split plans, saved-model inference and orchestration using synthetic fixtures. No model is fitted on the laptop. GPU speed, convergence and result quality require this Kaggle run.

Small participant coverage, protocol-driven recordings and unverified video alignment remain limitations. Seated/cooking classes have only two-person coverage. An extended training run cannot create missing diversity.
